# 11. Phase 5 최종검증 v2 — 제출서류 인용 복원 수정

유영관 · 2026-11-13

D02의 모델 응답은 제출서류 ①~⑤를 인용하면서 중간의 원본·평가용 표기 조건과 USB 수록 조건을 생략했습니다.
기존 코드는 생략 구간 하나만 분리할 수 있어 이 인용을 거절했습니다. 수정본은 같은 청크 안의 고유한 전체 행 구간을
최대 8개까지 **서로 다른 정확한 원문 인용**으로 분리합니다. 숫자·단위·문장부호 변경, 행 내부 생략, 다른 문서·청크,
순서 변경, 모호한 대응은 계속 거절합니다. 생략된 원문 조건도 CSV 검토 후보에 보존합니다.

**기존 11번 파일을 이 파일로 교체하고 커널을 Restart한 뒤 Run All하세요. 처음 API 호출은 0회입니다.**

1. 기본 Run All: 코드 백업·수정, 오프라인 테스트, 기존 D02 응답의 읽기 전용 캐시 재검증을 수행합니다.
   캐시에 실제 응답이 있으면 새로 생성하지 않습니다. ZIP에 있던 진단 인용만으로 원래 답변을 재구성하지 않습니다.
2. 재검증을 확인한 뒤 `ENABLE_OPENAI=True`, `RUN_DEV=True`로 Run All하세요.
   새 실행 ID는 `phase5_final_v2`입니다. 기존 v1 실패 기록과 답변은 보존합니다.
   같은 입력의 기존 정상 결과와 응답 캐시를 재사용합니다. D02 첫 배치 재사용에 성공해도 남은 5개 배치는 실제 생성이 필요합니다.
3. 개발 5건의 `comparison.csv`에서 정확성·누락없음·조건보존을 원문과 대조해 `통과` 또는 `실패`로 적고 `REVIEWER`에 이름을 입력합니다.
   `추가원문확인후보`에는 모델 인용에서 생략한 조건도 표시됩니다. 복원된 인용만으로 내용 검토가 통과되지는 않습니다.
4. 개발 5건 통과 후 `RUN_FINAL=True`로 최종 2건을 실행하고 검토하세요. 마지막 판정 셀에서 결과 ZIP을 만듭니다.

실패 응답은 자동 재시도하지 않습니다. 검토만 갱신할 때는 마지막 판정 셀만 실행하세요.
기존 문서·청크·인덱스·설정·응답 캐시는 변경하지 않습니다. 원문 기반 기준표는 모델 입력에 넣지 않습니다.


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Loads variables from .env into os.environ
load_dotenv()

# OpenAI() automatically looks for the "OPENAI_API_KEY" environment variable
client = OpenAI()

# Quick test call
response = client.responses.create(
    model="gpt-4o-mini",
    input="Say 'Environment loaded!' if this works."
)
print(response.output_text)

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

# LangChain also automatically reads OPENAI_API_KEY from the environment
llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [1]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import hashlib
import importlib.util
import json
import os
import sys

PROJECT_ROOT = None   # 자동 탐색 실패 시 기존 rfp_phase5_local_20261111 폴더의 경로
SUITE_ID = 'phase5_final_v3'
ENABLE_OPENAI = True
RUN_DEV = True
RUN_FINAL = False
REVIEWER = ''        # 원문과 결과를 검토한 사람의 이름

# 아래는 이번 검증의 호출·입력 byte 상한이며 달러 지출 상한이 아닙니다.
MAX_SUITE_CALLS = 75
MAX_SUITE_INPUT_BYTES = 2_400_000
PER_DOCUMENT_INPUT_CEILING = 900_000

if any(type(x) is not bool for x in (ENABLE_OPENAI, RUN_DEV, RUN_FINAL)):
    raise TypeError('실행 옵션은 True 또는 False로 지정하세요.')
if PROJECT_ROOT is None:
    candidates = [Path.cwd(), Path.cwd().parent,
        Path.cwd() / 'rfp_phase5_local_20261111', Path.cwd() / 'output/rfp_phase5_local_20261111']
    ROOT = next((p.resolve() for p in candidates if (p/'configs/phase5.json').is_file()
        and (p/'index/index_meta.json').is_file()), None)
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
if ROOT is None or not (ROOT/'configs/phase5.json').is_file():
    raise FileNotFoundError('PROJECT_ROOT에 기존 Phase 5 폴더 경로를 지정하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
for name, module in list(sys.modules.items()):
    if name.split('.')[0] in {'local_rag','rfp_phase4','rfp_phase5'} and getattr(module,'__file__',None):
        if not Path(module.__file__).resolve().is_relative_to(ROOT):
            raise RuntimeError('다른 Phase 폴더의 모듈이 로드됐습니다. 이 노트북 커널을 Restart하고 다시 실행하세요.')
required = ['rfp_phase5/diagnostics.py', 'rfp_phase5/source_audit.py',
            'tests/test_phase5_llm_diagnostics.py', 'tests/test_phase5_saved_diagnostics.py',
            'tests/test_phase5_source_audit.py']
if not all((ROOT/p).is_file() for p in required):
    raise RuntimeError('09번·10번 패치가 필요합니다. 해당 노트북을 각각 한 번 실행한 뒤 11번 커널을 Restart하세요.')
print('실행 중인 Python:', sys.executable)
print('프로젝트:', ROOT)
print('API 실행 허용:', ENABLE_OPENAI, '/ 개발:', RUN_DEV, '/ 최종:', RUN_FINAL)
print('검증 ID:', SUITE_ID)


실행 중인 Python: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase3_local_20261111/.venv/bin/python
프로젝트: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase5_local_20261111
API 실행 허용: True / 개발: True / 최종: False
검증 ID: phase5_final_v3


## 1. 수정 코드 적용 — 원래 파일 백업·버전 확인

인용 복원·요약 진단·최종검증 코드를 수정하고 회귀 테스트와 읽기 전용 캐시 재검증을 추가합니다.
허용된 이전 버전 또는 이번 수정본과 일치하는 파일만 처리합니다. 직접 수정한 파일이 있으면 덮어쓰지 않습니다.
기존 문서·인덱스·설정·캐시·v1 실행 결과는 보존합니다. API 호출은 없습니다.


In [ ]:
PATCH_ID = 'phase5-multifragment-citation-v3'
PATCH_FILES = json.loads('{"rfp_phase4/citation_recovery.py": "\\"\\"\\"Recover source quotations, then run the original strict validator.\\n\\nEvery non-whitespace character, the cited chunk, document, SHA and raw offsets\\nremain authoritative. This never paraphrases, changes digits/punctuation,\\nsearches other documents or invokes a model. Ambiguous recovery is rejected.\\nUp to eight uniquely matched whole-line source fragments are represented as\\nseparate exact citations, with all omitted gaps recorded for human review.\\nA false continuous quote is never kept or presented as semantic verification.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport copy\\nfrom functools import lru_cache\\nfrom datetime import datetime, timezone\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nimport re\\nimport sqlite3\\n\\n\\nRECOVERY_POLICY = \'unique_whitespace_or_bounded_whole_line_fragments_v3\'\\nMIN_SPLIT_FRAGMENT_CHARS = 12\\nMAX_SPLIT_FRAGMENTS = 8\\nMAX_SPLIT_QUOTE_LINES = 256\\nMAX_SPLIT_SOURCE_LINES = 2048\\n\\n\\ndef _compact_with_positions(text):\\n    positions = [i for i, char in enumerate(text) if not char.isspace()]\\n    return \'\'.join(text[i] for i in positions), positions\\n\\n\\ndef _recover_whitespace(source, quote):\\n    needle, _ = _compact_with_positions(quote)\\n    haystack, positions = _compact_with_positions(source)\\n    if not needle:\\n        return None, \'blank_quote\'\\n    start = haystack.find(needle)\\n    if start < 0:\\n        return None, \'non_whitespace_characters_differ\'\\n    if haystack.find(needle, start + 1) >= 0:\\n        return None, \'ambiguous_whitespace_match\'\\n    left, right = positions[start], positions[start + len(needle) - 1] + 1\\n    restored = source[left:right]\\n    assert \'\'.join(restored.split()) == \'\'.join(quote.split())\\n    return {\'quote\': restored, \'quote_start\': left, \'quote_end\': right}, \'whitespace_only\'\\n\\n\\ndef _source_lines(text):\\n    \\"\\"\\"Nonempty full lines with exact original character offsets.\\"\\"\\"\\n    result, offset = [], 0\\n    for raw in text.splitlines(keepends=True):\\n        positions = [i for i, char in enumerate(raw) if not char.isspace()]\\n        if positions:\\n            result.append({\'key\': \'\'.join(raw[i] for i in positions),\\n                           \'start\': offset + positions[0],\\n                           \'end\': offset + positions[-1] + 1})\\n        offset += len(raw)\\n    return result\\n\\n\\ndef _recover_two_fragments(source, quote):\\n    \\"\\"\\"Unique ordered full-line blocks, exactly one nonblank omitted source gap.\\n\\n    No character-level subsequence matching, intra-line omission, new words,\\n    cross-chunk search, reordered blocks or short isolated labels are accepted.\\n    \\"\\"\\"\\n    requested = [line[\'key\'] for line in _source_lines(quote)]\\n    lines = _source_lines(source)\\n    if len(requested) < 2:\\n        return None, \'not_a_multiline_whole_line_quote\'\\n    keys = [line[\'key\'] for line in lines]\\n    candidates = {}\\n    for split in range(1, len(requested)):\\n        first, second = requested[:split], requested[split:]\\n        if min(len(\'\'.join(first)), len(\'\'.join(second))) < MIN_SPLIT_FRAGMENT_CHARS:\\n            continue\\n        starts_a = [i for i in range(len(keys) - len(first) + 1) if keys[i:i+len(first)] == first]\\n        starts_b = [i for i in range(len(keys) - len(second) + 1) if keys[i:i+len(second)] == second]\\n        for a in starts_a:\\n            for b in starts_b:\\n                if b <= a + len(first):  # Require at least one nonempty omitted source line.\\n                    continue\\n                x, y = lines[a][\'start\'], lines[a+len(first)-1][\'end\']\\n                z, w = lines[b][\'start\'], lines[b+len(second)-1][\'end\']\\n                identity = x, y, z, w\\n                candidates[identity] = [\\n                    {\'quote\': source[x:y], \'quote_start\': x, \'quote_end\': y},\\n                    {\'quote\': source[z:w], \'quote_start\': z, \'quote_end\': w}]\\n                if len(candidates) > 1:\\n                    return None, \'ambiguous_fragment_mapping\'\\n    if not candidates:\\n        return None, \'no_unique_two_fragment_match\'\\n    fragments = next(iter(candidates.values()))\\n    assert \'\'.join(\'\'.join(f[\'quote\'].split()) for f in fragments) == \'\'.join(quote.split())\\n    return fragments, \'noncontiguous_quote_split\'\\n\\n\\ndef _recover_line_fragments(source, quote):\\n    \\"\\"\\"Unique ordered full-line blocks within one already selected chunk.\\n\\n    Every requested line must match a complete source line. No intra-line\\n    deletion, changed punctuation/digits, reordered text, cross-chunk search,\\n    or ambiguous placements are accepted. Consecutive source lines form one\\n    block; each block must contain at least twelve non-whitespace characters.\\n    \\"\\"\\"\\n    requested = [line[\'key\'] for line in _source_lines(quote)]\\n    lines = _source_lines(source)\\n    if len(requested) < 2:\\n        return None, \'not_a_multiline_whole_line_quote\'\\n    if len(requested) > MAX_SPLIT_QUOTE_LINES or len(lines) > MAX_SPLIT_SOURCE_LINES:\\n        return None, \'fragment_search_limit_exceeded\'\\n    keys = [line[\'key\'] for line in lines]\\n    positions = {}\\n    for i, key in enumerate(keys):\\n        positions.setdefault(key, []).append(i)\\n\\n    @lru_cache(maxsize=20000)\\n    def solve(qpos, minimum, remaining):\\n        if qpos == len(requested):\\n            return ((),)\\n        if remaining == 0:\\n            return ()\\n        found = []\\n        for start in positions.get(requested[qpos], ()):\\n            if start < minimum:\\n                continue\\n            length = 0\\n            while (qpos + length < len(requested) and start + length < len(keys)\\n                   and requested[qpos + length] == keys[start + length]):\\n                length += 1\\n            # Longer blocks first; all valid alternatives are still inspected.\\n            for count in range(length, 0, -1):\\n                if sum(len(key) for key in requested[qpos:qpos+count]) < MIN_SPLIT_FRAGMENT_CHARS:\\n                    continue\\n                end = start + count - 1\\n                for tail in solve(qpos + count, end + 2, remaining - 1):\\n                    candidate = ((start, end),) + tail\\n                    if candidate not in found:\\n                        found.append(candidate)\\n                    if len(found) > 1:\\n                        return tuple(found)\\n        return tuple(found)\\n\\n    matches = solve(0, 0, MAX_SPLIT_FRAGMENTS)\\n    if len(matches) > 1:\\n        return None, \'ambiguous_fragment_mapping\'\\n    if not matches or len(matches[0]) < 2:\\n        return None, \'no_unique_bounded_fragment_match\'\\n    fragments = [{\'quote\': source[lines[a][\'start\']:lines[b][\'end\']],\\n                  \'quote_start\': lines[a][\'start\'], \'quote_end\': lines[b][\'end\']}\\n                 for a, b in matches[0]]\\n    assert \'\'.join(\'\'.join(f[\'quote\'].split()) for f in fragments) == \'\'.join(quote.split())\\n    return fragments, \'noncontiguous_quote_split\'\\n\\n\\ndef validate_with_quote_recovery(registry, payload, context_chunks, allowed_doc_ids=None):\\n    \\"\\"\\"A formatting recovery layer; EvidenceRegistry itself remains strict.\\n\\n    Diagnostic quotations are unvalidated model data, never normal answer text.\\n    Cache payloads and caller objects are not mutated. A single unresolved error\\n    still prevents the whole model answer from being rendered.\\n    \\"\\"\\"\\n    strict = registry.validate_answer(payload, context_chunks, allowed_doc_ids=allowed_doc_ids)\\n    report = {\'policy\': RECOVERY_POLICY, \'corrected_citations\': 0, \'split_citations\': 0,\\n              \'additional_api_calls\': 0, \'strict_validation_initially_passed\': strict[\'valid\']}\\n    strict[\'citation_recovery\'] = report\\n    strict[\'citation_diagnostics\'] = []\\n    if strict[\'valid\']:\\n        return strict\\n    try:\\n        canonical = registry.validate_chunks(context_chunks, allowed_doc_ids=allowed_doc_ids)\\n    except (TypeError, ValueError, KeyError):\\n        return strict\\n    if not isinstance(payload, dict) or not isinstance(payload.get(\'claims\'), list):\\n        return strict\\n    repaired = copy.deepcopy(payload)\\n    context = {chunk[\'chunk_id\']: chunk for chunk in canonical}\\n    diagnostics = []\\n    corrections, splits = 0, 0\\n    for claim_index, claim in enumerate(repaired[\'claims\'], 1):\\n        if not isinstance(claim, dict) or not isinstance(claim.get(\'citations\'), list):\\n            continue\\n        replacement_citations = []\\n        for citation_index, citation in enumerate(claim[\'citations\'], 1):\\n            if not isinstance(citation, dict):\\n                replacement_citations.append(citation)\\n                continue\\n            replacement_citations.append(citation)\\n            cid, quote = citation.get(\'chunk_id\'), citation.get(\'quote\')\\n            item = {\'claim_index\': claim_index, \'citation_index\': citation_index,\\n                    \'claim_doc_id\': claim.get(\'doc_id\'), \'chunk_id\': cid,\\n                    \'model_quote\': quote, \'quote_is_model_data_not_a_verified_claim\': True}\\n            marker = re.match(r\'^\\\\[([a-z_]+)\\\\|(?:fact|notice_reference|conflict_candidate)\\\\] \', str(claim.get(\'text\', \'\')))\\n            item[\'claim_field_id\'] = marker.group(1) if marker else None\\n            chunk = context.get(cid) if isinstance(cid, str) else None\\n            if chunk is None:\\n                item[\'outcome\'] = \'chunk_not_in_supplied_context\'\\n            elif claim.get(\'doc_id\') != chunk[\'doc_id\']:\\n                item[\'outcome\'] = \'claim_and_chunk_document_differ\'\\n            elif not isinstance(quote, str) or not quote.strip():\\n                item[\'outcome\'] = \'invalid_quote\'\\n            elif quote in chunk[\'text\']:\\n                item[\'outcome\'] = \'already_exact\'\\n            else:\\n                recovered, outcome = _recover_whitespace(chunk[\'text\'], quote)\\n                item[\'outcome\'] = outcome\\n                if recovered is not None:\\n                    # Replacement is a literal source substring, not a normalized citation.\\n                    citation[\'quote\'] = recovered[\'quote\']\\n                    item[\'restored_source_quote\'] = recovered[\'quote\']\\n                    item[\'source_chunk_start\'] = recovered[\'quote_start\']\\n                    item[\'source_chunk_end\'] = recovered[\'quote_end\']\\n                    corrections += 1\\n                else:\\n                    item[\'cited_chunk_preview\'] = chunk[\'text\'][:300]\\n                    if outcome == \'non_whitespace_characters_differ\' and set(citation) == {\'chunk_id\',\'quote\'}:\\n                        fragments, split_outcome = _recover_line_fragments(chunk[\'text\'], quote)\\n                        item[\'split_check\'] = split_outcome\\n                        if fragments is not None:\\n                            replacement_citations.pop()\\n                            replacement_citations.extend({\'chunk_id\':cid,\'quote\':f[\'quote\']} for f in fragments)\\n                            item[\'outcome\'] = split_outcome\\n                            item[\'fragments\'] = fragments\\n                            gaps = []\\n                            for first, second in zip(fragments, fragments[1:]):\\n                                left, right = first[\'quote_end\'], second[\'quote_start\']\\n                                gaps.append({\'start\': left, \'end\': right,\\n                                             \'text\': chunk[\'text\'][left:right],\\n                                             \'preview\': chunk[\'text\'][left:right][:300]})\\n                            item[\'omitted_source_gap\'] = gaps[0]\\n                            item[\'omitted_source_gaps\'] = gaps\\n                            item[\'fragment_count\'] = len(fragments)\\n                            item[\'requires_conditions_review\'] = True\\n                            item[\'display_as_separate_quotes\'] = True\\n                            corrections += 1\\n                            splits += 1\\n            diagnostics.append(item)\\n        claim[\'citations\'] = replacement_citations\\n    validated = registry.validate_answer(repaired, canonical, allowed_doc_ids=allowed_doc_ids)\\n    validated[\'citation_recovery\'] = {**report, \'corrected_citations\': corrections, \'split_citations\':splits}\\n    validated[\'citation_diagnostics\'] = diagnostics\\n    return validated\\n\\n\\ndef _cached_response(root, saved):\\n    \\"\\"\\"Read only the matching cache row, with checksum and original input checks.\\"\\"\\"\\n    fingerprint = (saved.get(\'generation\') or {}).get(\'request_fingerprint\')\\n    if not isinstance(fingerprint, str) or len(fingerprint) != 64:\\n        raise ValueError(\'저장 결과에 요청 지문이 없습니다.\')\\n    cache_path = Path(root) / \'phase4_outputs/generation_cache/grounded_responses.sqlite3\'\\n    if not cache_path.is_file():\\n        raise FileNotFoundError(\'이 프로젝트의 기존 모델 응답 캐시가 없습니다. 추가 API 호출은 하지 않았습니다.\')\\n    with sqlite3.connect(cache_path.resolve().as_uri() + \'?mode=ro\', uri=True) as connection:\\n        row = connection.execute(\'SELECT request_json,response_json,checksum FROM responses WHERE fingerprint=?\',\\n                                 (fingerprint,)).fetchone()\\n    if row is None:\\n        raise ValueError(\'이 결과와 일치하는 캐시 응답이 없습니다. 추가 API 호출은 하지 않았습니다.\')\\n    if hashlib.sha256(row[0].encode()).hexdigest() != fingerprint:\\n        raise ValueError(\'캐시 요청 지문이 다릅니다.\')\\n    if hashlib.sha256((row[0] + \'\\\\n\' + row[1]).encode()).hexdigest() != row[2]:\\n        raise ValueError(\'캐시 응답 무결성 검사에 실패했습니다.\')\\n    request, response = json.loads(row[0]), json.loads(row[1])\\n    if response.get(\'request_fingerprint\') != fingerprint:\\n        raise ValueError(\'캐시 응답이 다른 요청을 가리킵니다.\')\\n    mode = response.get(\'execution_mode\')\\n    if mode not in {\'live_openai\',\'injected_test_client\'} or request.get(\'execution_namespace\') != mode:\\n        raise ValueError(\'캐시 요청과 응답의 실제/모의 실행 구분이 다릅니다.\')\\n    user = json.loads(request[\'request\'][\'input\'][1][\'content\'])\\n    expected_task = \'comparison\' if saved.get(\'retrieval\', {}).get(\'comparison\') else \'qa\'\\n    if user.get(\'task\') != expected_task or user.get(\'question\') != saved.get(\'retrieval\', {}).get(\'standalone_query\'):\\n        raise ValueError(\'캐시의 작업 또는 질문이 저장 결과와 다릅니다.\')\\n    evidence = user.get(\'evidence\', [])\\n    chunks = saved.get(\'source_chunks\', [])\\n    if len(evidence) != len(chunks):\\n        raise ValueError(\'캐시와 저장 결과의 입력 근거 개수가 다릅니다.\')\\n    for original, chunk in zip(evidence, chunks):\\n        for key in [\'chunk_id\',\'doc_id\',\'sha256\',\'text\']:\\n            if original.get(key) != chunk.get(key):\\n                raise ValueError(\'캐시와 저장 결과의 입력 근거가 다릅니다.\')\\n    generation = saved.get(\'generation\') or {}\\n    if response.get(\'execution_mode\') != generation.get(\'execution_mode\') or response.get(\'live_api\') != generation.get(\'live_api\'):\\n        raise ValueError(\'캐시 응답의 모의/실제 실행 구분이 다릅니다.\')\\n    return response\\n\\n\\ndef replay_saved_result(root, result_path, *, registry=None, persist=True):\\n    \\"\\"\\"Revalidate an existing response without API key, SDK or network calls.\\"\\"\\"\\n    root, path = Path(root).resolve(), Path(result_path).resolve()\\n    saved = json.loads(path.read_text(encoding=\'utf-8\'))\\n    response = _cached_response(root, saved)\\n    bootstrap_index = None\\n    if registry is None:\\n        from .runtime import bootstrap\\n        app = bootstrap(root)\\n        registry, bootstrap_index = app[\'registry\'], app[\'index\']\\n    try:\\n        checked = validate_with_quote_recovery(registry, response[\'payload\'], saved[\'source_chunks\'],\\n                                               allowed_doc_ids=saved.get(\'active_doc_ids\') or None)\\n        result = copy.deepcopy(saved)\\n        result[\'validation\'] = checked\\n        result[\'claims\'] = checked[\'claims\']\\n        result[\'missing_information\'] = checked[\'missing_information\']\\n        result[\'status\'] = checked[\'status\'] if checked[\'valid\'] else \'not_generated\'\\n        result[\'execution_status\'] = \'completed\' if checked[\'valid\'] else \'validation_failed\'\\n        result[\'answer_generated\'] = bool(checked[\'valid\'] and checked[\'claims\'])\\n        result[\'markdown\'] = registry.render(checked)\\n        if checked[\'valid\'] and saved.get(\'retrieval\', {}).get(\'comparison\'):\\n            from .rag import _comparison_markdown\\n            scopes = saved.get(\'active_doc_ids\') or []\\n            cited = {claim[\'doc_id\'] for claim in checked[\'claims\']}\\n            missing = [doc_id for doc_id in scopes if doc_id not in cited]\\n            if missing:\\n                checked[\'status\'] = \'partial\' if checked[\'claims\'] else \'insufficient_evidence\'\\n                checked[\'missing_information\'].extend(f\'선택 문서 {did}: 검증된 비교 근거 미확보\' for did in missing)\\n                result[\'status\'], result[\'missing_information\'] = checked[\'status\'],checked[\'missing_information\']\\n            catalog = [{\'doc_id\': did, \'file_name\': registry.documents_by_id[did].get(\'file_name\',did)}\\n                       for did in scopes]\\n            result[\'markdown\'] = _comparison_markdown(checked, scopes, catalog)\\n        result[\'replay\'] = {\'additional_api_calls\': 0, \'original_result_path\': str(path),\\n                            \'scope\': \'Offline source revalidation of the original cached model response; no new semantic quality measurement.\'}\\n        result[\'generation\'][\'original_usage\'] = response.get(\'original_usage\', response.get(\'usage\', {}))\\n        result[\'generation\'][\'usage\'] = {\'api_calls\':0,\'input_tokens\':0,\'output_tokens\':0,\\n                                          \'total_tokens\':0,\'cached_input_tokens\':0,\'usage_reported\':True}\\n        result[\'generation\'][\'cache_hit\'] = True\\n        result[\'generation\'][\'current_execution_mode\'] = \'offline_source_revalidation\'\\n        result[\'generation\'][\'request_latency_ms\'] = 0.0\\n        result[\'latency_ms\'] = None\\n        if persist:\\n            stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S_%fZ\')\\n            destination = path.with_name(path.stem + \'_revalidated_\' + stamp + \'.json\')\\n            with destination.open(\'x\',encoding=\'utf-8\') as handle:\\n                json.dump(result,handle,ensure_ascii=False,indent=2,allow_nan=False)\\n                handle.write(\'\\\\n\')\\n            result[\'revalidated_result_path\'] = str(destination)\\n        return result\\n    finally:\\n        if bootstrap_index is not None:\\n            bootstrap_index.close()\\n", "rfp_phase4/summarization.py": "\\"\\"\\"전체 문서의 모든 청크를 입력하는, 출처 검증 후 결정적으로 합치는 요약.\\n\\n입력 청크 100% 처리와 문서 의미 100% 요약은 별개다. 이 모듈은 전자를\\n계수하며 후자는 주장하지 않는다. 계획은 API 호출 없이 만들 수 있다.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport time\\nfrom collections.abc import Mapping\\n\\nfrom .evidence import ANSWER_SCHEMA\\nfrom .citation_recovery import validate_with_quote_recovery\\nfrom .llm import safe_llm_diagnostic\\n\\n\\nTASK = \'full_document_summary_map\'\\nDEFAULT_QUESTION = (\\n    \'이 RFP의 제공된 부분을 한국어로 항목별 정리해 주세요. 사업 개요, 예산과 부가세, \'\\n    \'사업기간, 참가자격, 요구사항, 제출 방식과 기한, 평가 기준, 산출물, \'\\n    \'계약상 유의사항을 살피세요. 각 사실에는 제공된 원문에서 정확히 복사한 인용문을 붙이세요. \'\\n    \'이 입력은 전체 문서의 일부일 수 있으므로 여기서 찾지 못한 항목을 문서 전체에 없다고 \'\\n    \'단정하지 마세요. 원문에 쓰인 금액·단위·조건·연도 차이를 그대로 보존하세요.\'\\n)\\nDEFAULT_CATEGORIES = (\\n    \'overview\', \'budget\', \'duration\', \'eligibility\', \'requirements\', \'submission\',\\n    \'evaluation\', \'deliverables\', \'risks\',\\n)\\nCATEGORY_LABELS = {\\n    \'overview\': \'사업 개요\', \'budget\': \'예산·부가세\', \'duration\': \'사업기간\',\\n    \'eligibility\': \'참가자격\', \'requirements\': \'요구사항\', \'submission\': \'제출 방식·기한\',\\n    \'evaluation\': \'평가 기준\', \'deliverables\': \'산출물\', \'risks\': \'계약상 유의사항\',\\n    \'other\': \'기타\',\\n}\\n\\n\\nclass SummaryError(ValueError):\\n    \\"\\"\\"잘못된 문서·배치 설정 또는 확인할 수 없는 입력의 오류.\\"\\"\\"\\n\\n\\ndef _field(value, key, default=None):\\n    return value.get(key, default) if isinstance(value, Mapping) else getattr(value, key, default)\\n\\n\\ndef _json(value):\\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(\',\', \':\'), allow_nan=False)\\n\\n\\nclass FullDocumentSummarizer:\\n    \\"\\"\\"모든 입력을 배치별로 요약하고 출처가 검증된 주장만 순서대로 합친다.\\n\\n    ``plan``은 request_input_bytes만 사용하며 네트워크를 호출하지 않는다.\\n    ``summarize``는 모든 요청의 예산·승인 목록을 먼저 검증한다. 기본 배치 상한은\\n    30회다. 큰 문서가 이를 넘으면 입력을 자르지 않고 차단 사유와 전체 계획을 돌려준다.\\n    \\"\\"\\"\\n\\n    def __init__(self, index, llm, registry, config=None):\\n        self.index, self.llm, self.registry = index, llm, registry\\n        supplied = dict(config or {})\\n        allowed = {\'max_map_calls\', \'max_batch_bytes\', \'batch_max_bytes\', \'requested_categories\', \'stop_on_error\'}\\n        if set(supplied) - allowed:\\n            raise SummaryError(\'지원하지 않는 전체 문서 요약 설정입니다.\')\\n        llm_config = getattr(llm, \'config\', {})\\n        self.max_map_calls = supplied.get(\'max_map_calls\', 30)\\n        request_limit = _field(llm_config, \'max_request_bytes\', 60000)\\n        if \'batch_max_bytes\' in supplied and \'max_batch_bytes\' in supplied:\\n            raise SummaryError(\'max_batch_bytes와 이전 별칭을 동시에 설정할 수 없습니다.\')\\n        self.batch_max_bytes = supplied.get(\'max_batch_bytes\', supplied.get(\'batch_max_bytes\', min(35000, request_limit)))\\n        if isinstance(self.batch_max_bytes, bool) or not isinstance(self.batch_max_bytes, int) or self.batch_max_bytes < 1:\\n            raise SummaryError(\'max_batch_bytes는 양의 정수여야 합니다.\')\\n        self.batch_max_bytes = min(self.batch_max_bytes, request_limit)\\n        self.max_output_tokens = _field(llm_config, \'max_output_tokens\', 3000)\\n        self.requested_categories = tuple(supplied.get(\'requested_categories\', DEFAULT_CATEGORIES))\\n        self.stop_on_error = supplied.get(\'stop_on_error\', True)\\n        for value in [self.max_map_calls, self.batch_max_bytes, self.max_output_tokens]:\\n            if isinstance(value, bool) or not isinstance(value, int) or value < 1:\\n                raise SummaryError(\'요약 배치·byte·출력 상한은 양의 정수여야 합니다.\')\\n        if not isinstance(self.stop_on_error, bool):\\n            raise SummaryError(\'stop_on_error는 bool이어야 합니다.\')\\n        if (not self.requested_categories or len(set(self.requested_categories)) != len(self.requested_categories)\\n                or any(value not in CATEGORY_LABELS for value in self.requested_categories)):\\n            raise SummaryError(\'requested_categories는 중복 없는 지원 항목 목록이어야 합니다.\')\\n        if not callable(getattr(llm, \'request_input_bytes\', None)):\\n            raise SummaryError(\'LLM의 순수 request_input_bytes 계획 함수가 필요합니다.\')\\n        if not callable(getattr(llm, \'preflight_requests\', None)):\\n            raise SummaryError(\'모든 배치의 사전 예산 검사 기능이 필요합니다.\')\\n\\n    def _request(self, question, chunks):\\n        return {\'task\': TASK, \'question\': question, \'evidence\': chunks, \'schema\': ANSWER_SCHEMA}\\n\\n    def _bytes(self, question, chunks):\\n        count = self.llm.request_input_bytes(TASK, question, chunks, schema=ANSWER_SCHEMA)\\n        if isinstance(count, bool) or not isinstance(count, int) or count < 1:\\n            raise SummaryError(\'요청 byte 계산이 잘못되었습니다.\')\\n        return count\\n\\n    def plan(self, doc_id, *, question=DEFAULT_QUESTION):\\n        \\"\\"\\"API 없이 전체 입력 계획을 만든다. 큰 문서도 모든 청크를 계획에 남긴다.\\"\\"\\"\\n        if not isinstance(question, str) or not question.strip():\\n            raise SummaryError(\'요약 질문을 입력해야 합니다.\')\\n        canonical = list(self.index.resolve_doc_ids([doc_id]))\\n        if len(canonical) != 1:\\n            raise SummaryError(\'전체 요약은 문서 한 개를 명확하게 선택해야 합니다.\')\\n        canonical = canonical[0]\\n        chunks = list(self.index.get_document_chunks(canonical))\\n        chunks = list(self.registry.validate_chunks(chunks, allowed_doc_ids=[canonical]))\\n        ids = [chunk[\'chunk_id\'] for chunk in chunks]\\n        if len(ids) != len(set(ids)):\\n            raise SummaryError(\'전체 문서 입력에 동일한 청크 ID가 중복됐습니다.\')\\n        if any(chunk[\'doc_id\'] != canonical for chunk in chunks):\\n            raise SummaryError(\'전체 문서 입력의 문서 범위가 다릅니다.\')\\n\\n        batches, block_reasons = [], []\\n\\n        def append_batch(items):\\n            if items:\\n                batches.append({\'batch_id\': f\'map-{len(batches) + 1:04d}\', \'chunks\': list(items),\\n                                \'request_bytes\': self._bytes(question, items)})\\n\\n        # Exponential growth followed by binary search keeps planning a large table-rich\\n        # document close to O(total serialized input) instead of repeatedly encoding every\\n        # growing prefix. Each batch is a consecutive interval of the original chunk order.\\n        offset = 0\\n        while offset < len(chunks):\\n            first_bytes = self._bytes(question, chunks[offset:offset + 1])\\n            if first_bytes > self.batch_max_bytes:\\n                append_batch(chunks[offset:offset + 1])\\n                block_reasons.append(\'one_or_more_source_chunks_exceed_request_byte_budget\')\\n                offset += 1\\n                continue\\n            remaining = len(chunks) - offset\\n            fits = 1\\n            probe = 2\\n            while probe <= remaining and self._bytes(question, chunks[offset:offset + probe]) <= self.batch_max_bytes:\\n                fits = probe\\n                probe *= 2\\n            low, high = fits + 1, min(probe, remaining)\\n            while low <= high:\\n                middle = (low + high) // 2\\n                if self._bytes(question, chunks[offset:offset + middle]) <= self.batch_max_bytes:\\n                    fits = middle\\n                    low = middle + 1\\n                else:\\n                    high = middle - 1\\n            append_batch(chunks[offset:offset + fits])\\n            offset += fits\\n        if not chunks:\\n            block_reasons.append(\'document_has_no_indexed_chunks\')\\n        if len(batches) > self.max_map_calls:\\n            block_reasons.append(\'max_map_calls_exceeded\')\\n        ordered = [chunk[\'chunk_id\'] for batch in batches for chunk in batch[\'chunks\']]\\n        if ordered != ids:\\n            raise SummaryError(\'전체 입력 계획에서 청크 순서 또는 개수가 유실됐습니다.\')\\n        total_bytes = sum(batch[\'request_bytes\'] for batch in batches)\\n        fingerprint = hashlib.sha256(_json({\'doc_id\': canonical, \'question\': question,\\n             \'source_chunks\': [(chunk[\'chunk_id\'], chunk[\'sha256\'], hashlib.sha256(chunk[\'text\'].encode(\'utf-8\')).hexdigest()) for chunk in chunks],\\n             \'batches\': [[chunk[\'chunk_id\'] for chunk in batch[\'chunks\']] for batch in batches],\\n             \'requested_categories\': self.requested_categories, \'batch_max_bytes\': self.batch_max_bytes,\\n             \'max_output_tokens\': self.max_output_tokens}).encode(\'utf-8\')).hexdigest()\\n        return {\'doc_id\': canonical, \'question\': question, \'source_chunk_ids\': ids,\\n                \'chunk_count\': len(chunks), \'map_calls\': len(batches), \'batches\': batches,\\n                \'total_request_bytes\': total_bytes,\\n                \'uncached_input_bytes_upper_bound\': total_bytes,\\n                \'llm_max_calls\': _field(getattr(self.llm, \'config\', {}), \'max_calls\'),\\n                \'llm_max_total_input_bytes\': _field(getattr(self.llm, \'config\', {}), \'max_total_input_bytes\'),\\n                \'whole_session_budget_preflight_required\': True,\\n                \'max_output_tokens_upper_bound\': len(batches) * self.max_output_tokens,\\n                \'byte_budget_is_token_count\': False, \'estimated_price\': None,\\n                \'max_map_calls\': self.max_map_calls, \'max_batch_bytes\': self.batch_max_bytes,\\n                \'batch_max_bytes\': self.batch_max_bytes,\\n                \'can_execute\': not block_reasons, \'block_reasons\': list(dict.fromkeys(block_reasons)),\\n                \'plan_fingerprint\': fingerprint, \'api_calls\': 0,\\n                \'scope\': \'Every indexed chunk appears once, in source order. This is an input plan, not semantic summary coverage.\'}\\n\\n    def _empty_result(self, plan):\\n        return {\'status\': \'blocked\', \'execution_status\': \'not_started\', \'doc_id\': plan[\'doc_id\'],\\n                \'valid\': False, \'claims\': [], \'missing_information\': [], \'errors\': [],\\n                \'answer_generated\': False, \'markdown\': \'\',\\n                \'source_chunk_ids\': plan[\'source_chunk_ids\'], \'processed_chunk_ids\': [],\\n                \'response_received_chunk_ids\': [], \'request_attempted_chunk_ids\': [],\\n                \'input_chunk_coverage\': 0.0, \'validated_chunk_coverage\': 0.0,\\n                \'batch_outcomes\': [], \'map_calls\': plan[\'map_calls\'],\\n                \'plan_fingerprint\': plan[\'plan_fingerprint\'], \'preflight\': None,\\n                \'aggregation_method\': \'deterministic_merge_of_source_validated_map_claims\',\\n                \'semantic_coverage_verified\': False, \'semantic_entailment_verified\': False,\\n                \'answer_correctness_verified\': False,\\n                \'usage\': {\'api_calls\': 0, \'live_api_calls\': 0, \'mock_calls\': 0,\\n                          \'input_tokens\': 0, \'output_tokens\': 0, \'total_tokens\': 0,\\n                          \'usage_unknown_attempts\': 0},\\n                \'input_coverage_definition\': \'Fraction of indexed chunks with a completed model/cache response. Validated coverage separately requires valid source citations.\',\\n                \'scope\': \'Full-document map input and citation provenance only; semantic accuracy/completeness needs review.\'}\\n\\n    @staticmethod\\n    def _merge_claims(claims):\\n        \\"\\"\\"Only exact duplicate claim text is merged; no uncited synthesis is invented.\\"\\"\\"\\n        merged, by_key = [], {}\\n        for claim in claims:\\n            key = claim[\'doc_id\'], claim[\'category\'], claim[\'text\']\\n            if key not in by_key:\\n                fresh = {**claim, \'citations\': [dict(citation) for citation in claim[\'citations\']]}\\n                by_key[key] = fresh\\n                merged.append(fresh)\\n            else:\\n                target = by_key[key][\'citations\']\\n                known = {(citation[\'chunk_id\'], citation[\'quote\']) for citation in target}\\n                for citation in claim[\'citations\']:\\n                    pair = citation[\'chunk_id\'], citation[\'quote\']\\n                    if pair not in known:\\n                        target.append(dict(citation)); known.add(pair)\\n        return merged\\n\\n    def summarize(self, doc_id, question=DEFAULT_QUESTION):\\n        started = time.perf_counter()\\n        plan = self.plan(doc_id, question=question)\\n        result = self._empty_result(plan)\\n        usage_before = self.llm.usage_report() if callable(getattr(self.llm, \'usage_report\', None)) else {}\\n        if not plan[\'can_execute\']:\\n            result[\'execution_status\'] = \'plan_blocked\'\\n            result[\'errors\'] = plan[\'block_reasons\']\\n            result[\'markdown\'] = \'전체 문서 요약을 실행하지 않았습니다. 입력을 자르지 않고 계획 상한 초과를 반환했습니다.\'\\n            result[\'latency_ms\'] = (time.perf_counter() - started) * 1000\\n            return result\\n        requests = [self._request(question, batch[\'chunks\']) for batch in plan[\'batches\']]\\n        try:\\n            result[\'preflight\'] = self.llm.preflight_requests(requests)\\n            if not isinstance(result[\'preflight\'], dict) or result[\'preflight\'].get(\'valid\') is not True:\\n                raise SummaryError(\'전체 요청의 사전 검사가 통과하지 않았습니다.\')\\n        except Exception as error:\\n            result[\'execution_status\'] = \'preflight_blocked\'\\n            result[\'errors\'] = [\'whole_document_preflight_failed:\' + type(error).__name__]\\n            result[\'error_diagnostics\'] = [safe_llm_diagnostic(error)]\\n            result[\'error_detail\'] = result[\'error_diagnostics\'][0][\'message\']\\n            result[\'markdown\'] = \'전체 문서 요약을 실행하지 않았습니다. 활성화·문서 승인·전체 요청 예산을 확인하세요.\'\\n            result[\'latency_ms\'] = (time.perf_counter() - started) * 1000\\n            return result\\n\\n        collected, processed, responded, attempted, outcomes = [], [], [], [], []\\n        halted = False\\n        for batch in plan[\'batches\']:\\n            chunk_ids = [chunk[\'chunk_id\'] for chunk in batch[\'chunks\']]\\n            outcome = {\'batch_id\': batch[\'batch_id\'], \'chunk_ids\': chunk_ids,\\n                       \'request_bytes\': batch[\'request_bytes\'], \'status\': \'skipped\',\\n                       \'valid\': False, \'usage\': {}, \'live_api\': False, \'cache_hit\': False,\\n                       \'api_called\': False}\\n            if halted:\\n                outcome[\'reason\'] = \'previous_batch_failed\'\\n                outcomes.append(outcome)\\n                continue\\n            attempted.extend(chunk_ids)\\n            try:\\n                response = self.llm.generate(TASK, question, batch[\'chunks\'], schema=ANSWER_SCHEMA)\\n                responded.extend(chunk_ids)\\n                outcome.update({\'usage\': response.get(\'usage\', {}), \'live_api\': bool(response.get(\'live_api\')),\\n                                \'cache_hit\': bool(response.get(\'cache_hit\')),\\n                                \'api_called\': bool(response.get(\'live_api\')) and not bool(response.get(\'cache_hit\')),\\n                                \'execution_mode\': response.get(\'execution_mode\', \'mock_or_unspecified\'),\\n                                \'request_fingerprint\': response.get(\'request_fingerprint\')})\\n                validated = validate_with_quote_recovery(self.registry, response[\'payload\'], batch[\'chunks\'],\\n                                                          allowed_doc_ids=[plan[\'doc_id\']])\\n                # Keep recovery and omitted-source diagnostics for accepted batches too.\\n                outcome[\'citation_recovery\'] = validated.get(\'citation_recovery\', {})\\n                outcome[\'citation_diagnostics\'] = validated.get(\'citation_diagnostics\', [])\\n                if validated.get(\'valid\') is not True:\\n                    outcome.update({\'status\': \'invalid\', \'reason\': \'source_citation_validation_failed\',\\n                                    \'validation_errors\':validated[\'errors\'],\\n                                    \'citation_diagnostics\':validated.get(\'citation_diagnostics\',[])})\\n                    result[\'errors\'].append(batch[\'batch_id\'] + \':source_citation_validation_failed\')\\n                    halted = self.stop_on_error\\n                else:\\n                    outcome.update({\'status\': validated[\'status\'], \'valid\': True,\\n                                    \'claim_count\': len(validated[\'claims\']),\\n                                    \'batch_missing_information\': validated.get(\'missing_information\', [])})\\n                    processed.extend(chunk_ids)\\n                    collected.extend(validated[\'claims\'])\\n            except Exception as error:\\n                outcome.update({\'status\': \'error\', \'reason\': \'batch_failed:\' + type(error).__name__})\\n                diagnostic = safe_llm_diagnostic(error)\\n                outcome[\'error_diagnostic\'] = diagnostic\\n                outcome[\'error_detail\'] = diagnostic[\'message\']\\n                result.setdefault(\'error_diagnostics\', []).append({\'batch_id\': batch[\'batch_id\'], **diagnostic})\\n                result[\'errors\'].append(batch[\'batch_id\'] + \':batch_failed:\' + type(error).__name__)\\n                halted = self.stop_on_error\\n            outcomes.append(outcome)\\n\\n        claims = self._merge_claims(collected)\\n        covered_categories = {claim[\'category\'] for claim in claims}\\n        missing = [CATEGORY_LABELS[category] + \': 검증된 근거를 확보하지 못했습니다.\'\\n                   for category in self.requested_categories if category not in covered_categories]\\n        # A category label does not resolve a finer uncertainty such as whether VAT is\\n        # included. Without a semantic reduction/review step, preserve each map\'s own\\n        # uncertainty conservatively and keep its limited-input scope visible.\\n        partial_batches = [outcome for outcome in outcomes if outcome.get(\'valid\') and outcome[\'status\'] == \'partial\']\\n        for outcome in outcomes:\\n            if not outcome.get(\'valid\'):\\n                continue\\n            for label in outcome.get(\'batch_missing_information\', []):\\n                missing.append(f\'부분 입력 {outcome[\\"batch_id\\"]}에서 미확인: {label}\')\\n        if partial_batches:\\n            missing.append(\'부분 입력의 미확인 사항이 다른 입력에서 해결됐는지는 자동으로 검증하지 않았습니다.\')\\n        missing = list(dict.fromkeys(missing))\\n        all_valid = len(processed) == len(plan[\'source_chunk_ids\'])\\n        if not all_valid:\\n            missing.append(\'일부 원문 청크를 검증된 요약으로 처리하지 못했습니다. 전체 문서 요약으로 확정할 수 없습니다.\')\\n        if not claims:\\n            status = \'insufficient_evidence\' if all_valid else \'error\'\\n        else:\\n            status = \'answered\' if all_valid and not missing else \'partial\'\\n        result.update({\'status\': status, \'execution_status\': \'completed\' if all_valid else \'incomplete\',\\n                       \'valid\': bool(claims) or all_valid, \'claims\': claims, \'missing_information\': missing,\\n                       \'processed_chunk_ids\': processed, \'response_received_chunk_ids\': responded,\\n                       \'request_attempted_chunk_ids\': attempted, \'batch_outcomes\': outcomes,\\n                       \'partial_batch_count\': len(partial_batches),\\n                       \'input_chunk_coverage\': len(responded) / plan[\'chunk_count\'],\\n                       \'validated_chunk_coverage\': len(processed) / plan[\'chunk_count\'],\\n                       \'answer_generated\': any(outcome[\'valid\'] for outcome in outcomes),\\n                       \'elapsed_seconds\': time.perf_counter() - started})\\n        if result.get(\'error_diagnostics\'):\\n            result[\'error_detail\'] = \' / \'.join(dict.fromkeys(item[\'message\'] for item in result[\'error_diagnostics\']))\\n        if result[\'valid\']:\\n            rendered = self.registry.render(result)\\n            result[\'markdown\'] = rendered + \'\\\\n\\\\n입력 청크 처리율은 의미적 요약 완전성이나 사실 정확도를 보장하지 않습니다.\'\\n        else:\\n            result[\'markdown\'] = \'검증된 전체 문서 요약을 만들지 못했습니다. 처리 결과와 누락 청크를 확인하세요.\'\\n        result[\'latency_ms\'] = (time.perf_counter() - started) * 1000\\n        if callable(getattr(self.llm, \'usage_report\', None)):\\n            after = self.llm.usage_report()\\n            result[\'llm_session_usage\'] = after\\n            for name in [\'api_calls\', \'live_api_calls\', \'mock_calls\', \'usage_unknown_attempts\']:\\n                result[\'usage\'][name] = (after[name] - usage_before.get(name, 0)\\n                                         if isinstance(after.get(name), (int, float)) else None)\\n            for name in [\'input_tokens\', \'output_tokens\', \'total_tokens\', \'cached_input_tokens\']:\\n                key = \'reported_\' + name\\n                result[\'usage\'][name] = (after[key] - usage_before.get(key, 0)\\n                                        if isinstance(after.get(key), (int, float)) else None)\\n            result[\'usage\'][\'scope\'] = \'Counters for this summary invocation; failed attempts may have unknown token usage.\'\\n        return result\\n", "tests/test_phase4_citation_split.py": "\\"\\"\\"A joined noncontiguous quote becomes two exact quotes, never a fake substring.\\"\\"\\"\\nimport copy\\nimport unittest\\n\\nfrom test_phase4_evidence import block,document,chunk,answer\\nfrom rfp_phase4.evidence import EvidenceRegistry\\nfrom rfp_phase4.citation_recovery import validate_with_quote_recovery\\n\\n\\nHEADER=\'<부가세 포함, 금액단위 : 원>\'\\nBODY=\'구       분\\\\n금      액\\\\n비     고\\\\n용 역 비\\\\n부가가치세 포함\\\\n장  비\\\\n구입비\\\\nHW 구매 부문\\\\n부가가치세 포함\'\\nGAP=\'발주자\\\\n한국주택금융공사\\\\n직전도급자\\\\n-\\\\n하도급\\\\n주는자\\\\n사업명\\\\n계약금액\\\\n상  호\\\\n대표자\'\\n\\n\\nclass CitationSplitTests(unittest.TestCase):\\n    def check(self,source,quote,*,did=\'d1\',allowed=None):\\n        b=block(text=source);d=document(source=b);c=chunk(b)\\n        reg=EvidenceRegistry([d],[c]);payload=answer(quote=quote,did=did)\\n        original=copy.deepcopy(payload)\\n        before=reg.validate_answer(payload,[c],allowed)\\n        after=validate_with_quote_recovery(reg,payload,[c],allowed)\\n        self.assertEqual(payload,original)\\n        return before,after,d,c\\n\\n    def test_reported_header_plus_table_body_is_two_exact_citations(self):\\n        before,r,d,c=self.check(HEADER+\'\\\\n\'+GAP+\'\\\\n\'+BODY+\'\\\\nSW 구매 부문\\\\n부가가치세 포함\',HEADER+\'\\\\n\'+BODY)\\n        self.assertFalse(before[\'valid\']);self.assertTrue(r[\'valid\'])\\n        citations=r[\'claims\'][0][\'citations\']\\n        self.assertEqual(len(citations),2)\\n        self.assertEqual([v[\'quote\'] for v in citations],[HEADER,BODY])\\n        self.assertEqual(r[\'citation_recovery\'][\'split_citations\'],1)\\n        self.assertEqual(r[\'citation_recovery\'][\'additional_api_calls\'],0)\\n        for v in citations:\\n            self.assertIn(v[\'quote\'],c[\'text\'])\\n            for span in v[\'source_ranges\']:\\n                self.assertEqual(d[\'blocks\'][0][\'text\'][span[\'char_start\']:span[\'char_end\']],\\n                                 c[\'text\'][span[\'chunk_start\']:span[\'chunk_end\']])\\n        self.assertNotIn(GAP,\' \'.join(v[\'quote\'] for v in citations))\\n\\n    def test_whitespace_inside_each_fragment_is_restored(self):\\n        _,r,_,_=self.check(HEADER+\'\\\\n\'+GAP+\'\\\\n\'+BODY,HEADER.replace(\' \',\'\')+\'\\\\n\'+\'\\\\n\'.join(s.replace(\' \',\'\') for s in BODY.splitlines()))\\n        self.assertTrue(r[\'valid\']);self.assertEqual(r[\'claims\'][0][\'citations\'][1][\'quote\'],BODY)\\n\\n    def test_changed_numbers_vat_units_punctuation_and_words_are_rejected(self):\\n        source=HEADER+\'\\\\n\'+GAP+\'\\\\n사업예산은 688백만원이며 부가세 포함입니다.\'\\n        for tail in [\'사업예산은 699백만원이며 부가세 포함입니다.\',\'사업예산은 688백만원이며 부가세 별도입니다.\',\\n                     \'사업예산은 688달러이며 부가세 포함입니다.\',\'사업예산은 688백만원이며 부가세 포함입니다!\',\\n                     \'사업예산은 688백만원으로 확정되었습니다.\']:\\n            self.assertFalse(self.check(source,HEADER+\'\\\\n\'+tail)[1][\'valid\'])\\n\\n    def test_intra_line_omission_and_negation_removal_are_rejected(self):\\n        source=HEADER+\'\\\\n\'+GAP+\'\\\\n사업예산은 688백만원이며 부가세 포함이 아닙니다.\'\\n        r=self.check(source,HEADER+\'\\\\n사업예산은 688백만원이며 부가세 포함입니다.\')[1]\\n        self.assertFalse(r[\'valid\'])\\n\\n    def test_reordered_fragments_are_rejected(self):\\n        self.assertFalse(self.check(HEADER+\'\\\\n\'+GAP+\'\\\\n\'+BODY,BODY+\'\\\\n\'+HEADER)[1][\'valid\'])\\n\\n    def test_duplicate_fragment_placements_are_ambiguous(self):\\n        source=HEADER+\'\\\\n\'+GAP+\'\\\\n\'+BODY+\'\\\\n여백\\\\n\'+BODY\\n        r=self.check(source,HEADER+\'\\\\n\'+BODY)[1]\\n        self.assertFalse(r[\'valid\'])\\n        self.assertEqual(r[\'citation_diagnostics\'][0][\'split_check\'],\'ambiguous_fragment_mapping\')\\n\\n    def test_several_omitted_blocks_become_separate_exact_citations(self):\\n        a=\'사업예산은 688백만원입니다.\';b=\'사업기간은 계약일부터 5개월입니다.\';c=\'안정화 기간은 1개월 포함입니다.\'\\n        source=a+\'\\\\n생략 구간 하나\\\\n\'+b+\'\\\\n생략 구간 둘\\\\n\'+c\\n        r=self.check(source,a+\'\\\\n\'+b+\'\\\\n\'+c)[1]\\n        self.assertTrue(r[\'valid\'])\\n        self.assertEqual([q[\'quote\'] for q in r[\'claims\'][0][\'citations\']],[a,b,c])\\n        self.assertEqual(len(r[\'citation_diagnostics\'][0][\'omitted_source_gaps\']),2)\\n        self.assertFalse(r[\'semantic_entailment_verified\'])\\n\\n    def test_eight_fragments_are_bounded_and_ninth_is_rejected(self):\\n        pieces=[f\'제출서류 {i}번의 원본과 사본을 모두 제출해야 합니다.\' for i in range(9)]\\n        self.assertTrue(self.check(\'\\\\n검토할 생략 조건\\\\n\'.join(pieces[:8]),\'\\\\n\'.join(pieces[:8]))[1][\'valid\'])\\n        self.assertFalse(self.check(\'\\\\n검토할 생략 조건\\\\n\'.join(pieces),\'\\\\n\'.join(pieces))[1][\'valid\'])\\n\\n    def test_intra_line_omission_stays_rejected_with_three_fragments(self):\\n        a=\'제안서 원본은 반드시 제출해야 합니다.\'\\n        b=\'가격제안서는 별도로 밀봉한 경우에만 접수합니다.\'\\n        c=\'제출서류는 원본대조필을 반드시 명기해야 합니다.\'\\n        source=a+\'\\\\n누락구간 하나\\\\n\'+b+\'\\\\n누락구간 둘\\\\n\'+c\\n        self.assertFalse(self.check(source,a+\'\\\\n가격제안서는 접수합니다.\\\\n\'+c)[1][\'valid\'])\\n\\n    def test_short_middle_fragment_cannot_bridge_long_blocks(self):\\n        a=\'사업예산은 650백만원으로 명시되어 있습니다.\'\\n        c=\'사업기간은 계약일부터 10개월로 명시되어 있습니다.\'\\n        self.assertFalse(self.check(a+\'\\\\n중간 조건 하나\\\\n필수\\\\n중간 조건 둘\\\\n\'+c,a+\'\\\\n필수\\\\n\'+c)[1][\'valid\'])\\n\\n    def test_repeated_third_fragment_remains_ambiguous(self):\\n        a=\'원본 제안서는 반드시 한 부를 제출합니다.\'\\n        b=\'발표자료는 반드시 열 부를 제출합니다.\'\\n        c=\'가격제안서는 별도로 밀봉하여 제출합니다.\'\\n        source=a+\'\\\\n원본 조건\\\\n\'+b+\'\\\\n평가 조건\\\\n\'+c+\'\\\\n반복 구간\\\\n\'+c\\n        self.assertFalse(self.check(source,a+\'\\\\n\'+b+\'\\\\n\'+c)[1][\'valid\'])\\n\\n    def test_omitted_negation_is_visible_and_never_semantic_pass(self):\\n        a=\'제안서 제출 방식은 직접 방문 접수입니다.\'\\n        b=\'참가 신청 서류는 원본을 제출해야 합니다.\'\\n        c=\'가격입찰 서류는 별도로 제출해야 합니다.\'\\n        source=a+\'\\\\n전자우편 제출은 허용되지 않습니다.\\\\n\'+b+\'\\\\n접수기한 이후 제출할 수 없습니다.\\\\n\'+c\\n        r=self.check(source,a+\'\\\\n\'+b+\'\\\\n\'+c)[1]\\n        self.assertTrue(r[\'valid\'])\\n        diag=r[\'citation_diagnostics\'][0]\\n        self.assertTrue(diag[\'requires_conditions_review\'])\\n        self.assertIn(\'허용되지 않습니다\',diag[\'omitted_source_gaps\'][0][\'preview\'])\\n        self.assertFalse(r[\'semantic_entailment_verified\'])\\n\\n    def test_fragment_search_refuses_oversized_input(self):\\n        a=\'사업예산은 650백만원이며 부가세가 포함됩니다.\'\\n        source=a+\'\\\\n\'+\'중복 아닌 설명 줄\\\\n\'*2050+\'사업기간은 계약일부터 10개월입니다.\'\\n        quote=a+\'\\\\n사업기간은 계약일부터 10개월입니다.\'\\n        self.assertFalse(self.check(source,quote)[1][\'valid\'])\\n\\n    def test_short_isolated_labels_do_not_enable_split(self):\\n        source=\'예산\\\\n다른 문장\\\\n부가세는 포함이며 예산은 688백만원입니다.\'\\n        self.assertFalse(self.check(source,\'예산\\\\n부가세는 포함이며 예산은 688백만원입니다.\')[1][\'valid\'])\\n\\n    def test_scope_and_document_checks_still_apply(self):\\n        source=HEADER+\'\\\\n\'+GAP+\'\\\\n\'+BODY;quote=HEADER+\'\\\\n\'+BODY\\n        self.assertFalse(self.check(source,quote,allowed=[])[1][\'valid\'])\\n        self.assertFalse(self.check(source,quote,did=\'other\')[1][\'valid\'])\\n\\n    def test_true_contiguous_quote_is_not_split(self):\\n        _,r,_,_=self.check(HEADER+\'\\\\n\'+BODY,HEADER+\'\\\\n\'+BODY)\\n        self.assertTrue(r[\'valid\']);self.assertEqual(r[\'citation_recovery\'][\'split_citations\'],0)\\n        self.assertEqual(len(r[\'claims\'][0][\'citations\']),1)\\n\\n    def test_replaying_revalidated_output_keeps_original_usage(self):\\n        import test_phase4_citation_recovery as fixtures\\n        from rfp_phase4.citation_recovery import replay_saved_result\\n        fixture=fixtures.ReplayTests();fixture.setUp();self.addCleanup(fixture.doCleanups)\\n        first=replay_saved_result(fixture.root,fixture.path,registry=fixture.reg)\\n        second=replay_saved_result(fixture.root,first[\'revalidated_result_path\'],registry=fixture.reg)\\n        self.assertEqual(second[\'generation\'][\'original_usage\'][\'api_calls\'],1)\\n        self.assertEqual(second[\'generation\'][\'usage\'][\'api_calls\'],0)\\n\\n\\n    def test_reported_D02_form_list_is_three_exact_source_fragments(self):\\n        source=\'구분\\\\n제출장소\\\\n제출서류\\\\n비고\\\\n제안서\\\\n제  출\\\\n뷰티패션\\\\n산업과\\\\n(서소문2청사 18층)\\\\n① 제안서 제출 공문 1부\\\\n<서식 6호>\\\\n② 사업수행능력평가서 1부\\\\n- 제안사 일반현황 1부\\\\n- 사업수행 조직 및 역할분담 체계 1부\\\\n- 참여인력 현황 1부\\\\n- 참여인력 이력사항 1부\\\\n- 용역 수행실적(원본) 1부\\\\n- 실적증명서 원본 각 1부\\\\n- 제안사 경영상태 1부(신용평가등급확인서 등)\\\\n-서약서 1부\\\\n- 정량적 평가지표 자가진단표 1부\\\\n<서식 7~16호>\\\\n③ 사업내용제안서 및 제안서 요약본(발표자료) 각 11부 (원본 각1부, 평가용 각10부)\\\\n※ 원 본  1부  : 표지에 제안기관명, 대표자명 기재 후 인감날인\\\\n평가용 10부 : 표지에 제안기관명, 대표자명 기재하지 않고 빈칸 처리\\\\n<서식\\\\n17호>\\\\n④ 사업내용제안서 및 발표자료 수록 USB 1개\\\\n※ 제안서 원본(PDF), 발표자료(PPT 및 PDF) 수록\\\\n-\\\\n⑤ 가격제안서(산출내역서 포함) 1부\\\\n※ 별도 밀봉 후 인감날인\\\\n<서식 18,19호>\'\\n        quote=\'① 제안서 제출 공문 1부\\\\n<서식 6호>\\\\n② 사업수행능력평가서 1부\\\\n- 제안사 일반현황 1부\\\\n- 사업수행 조직 및 역할분담 체계 1부\\\\n- 참여인력 현황 1부\\\\n- 참여인력 이력사항 1부\\\\n- 용역 수행실적(원본) 1부\\\\n- 실적증명서 원본 각 1부\\\\n- 제안사 경영상태 1부(신용평가등급확인서 등)\\\\n-서약서 1부\\\\n- 정량적 평가지표 자가진단표 1부\\\\n<서식 7~16호>\\\\n③ 사업내용제안서 및 제안서 요약본(발표자료) 각 11부 (원본 각1부, 평가용 각10부)\\\\n④ 사업내용제안서 및 발표자료 수록 USB 1개\\\\n⑤ 가격제안서(산출내역서 포함) 1부\'\\n        before,r,d,c=self.check(source,quote)\\n        self.assertFalse(before[\'valid\'])\\n        self.assertTrue(r[\'valid\'])\\n        self.assertEqual(r[\'citation_recovery\'][\'additional_api_calls\'],0)\\n        pieces=r[\'claims\'][0][\'citations\']\\n        self.assertEqual(len(pieces),3)\\n        self.assertEqual(\'\'.join(\'\'.join(p[\'quote\'].split()) for p in pieces),\'\'.join(quote.split()))\\n        self.assertEqual(len(r[\'citation_diagnostics\'][0][\'omitted_source_gaps\']),2)\\n        for p in pieces:\\n            self.assertIn(p[\'quote\'],source)\\n            self.assertTrue(p[\'provenance_verified\'])\\n        self.assertFalse(r[\'answer_correctness_verified\'])\\n        for wrong in [quote.replace(\'각 11부\',\'각 10부\'),quote.replace(\'USB 1개\',\'USB 2개\'),quote.replace(\'1부\',\'2부\',1)]:\\n            self.assertFalse(self.check(source,wrong)[1][\'valid\'])\\n\\n\\nif __name__==\'__main__\':unittest.main()\\n", "phase5_validation/final_validation.py": "\\"\\"\\"Phase 5 sample acceptance. No network except an explicitly enabled run_stage.\\n\\nThe runner never changes source/index/config files or rewrites a model answer.\\nReference drafts are used only in local human review, never sent to the model.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport copy\\nimport csv\\nfrom dataclasses import asdict\\nfrom datetime import datetime, timezone\\nimport hashlib\\nimport io\\nimport json\\nfrom pathlib import Path\\nimport re\\nimport subprocess\\nimport sys\\nimport zipfile\\n\\nVERSION = \'phase5-final-validation-v2\'\\nJUDGEMENTS = (\'정확성\', \'누락없음\', \'조건보존\')\\nBASE_COLUMNS = (\'항목ID\', \'항목\', \'기준내용\', \'필수조건\', \'실제요약\', \'확인상태\', \'기준원문위치\', \'실제인용\', \'추가원문확인후보\')\\nCORE_FILES = (\'rfp_phase4/llm.py\', \'rfp_phase4/evidence.py\',\\n              \'rfp_phase4/citation_recovery.py\', \'rfp_phase4/summarization.py\',\\n              \'rfp_phase5/fields.py\', \'rfp_phase5/summary.py\',\\n              \'rfp_phase5/source_audit.py\', \'rfp_phase5/quality.py\',\\n              \'rfp_phase5/runtime.py\', \'rfp_phase5/review.py\', \'rfp_phase4/runtime.py\',\\n              \'local_rag/io.py\', \'local_rag/retrieval.py\', \'phase5_validation/cache_revalidation.py\',\\n              \'tests/test_phase5_summary.py\', \'tests/test_phase5_llm_diagnostics.py\',\\n              \'tests/test_phase5_saved_diagnostics.py\', \'tests/test_phase5_source_audit.py\',\\n              \'tests/test_phase4_citation_recovery.py\', \'tests/test_phase4_citation_split.py\',\\n              \'phase5_validation/test_final_validation.py\', \'phase5_validation/test_cache_revalidation.py\')\\n\\n\\ndef digest(value):\\n    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True,\\n        separators=(\',\', \':\'), allow_nan=False).encode()).hexdigest()\\n\\n\\ndef file_hash(path):\\n    h = hashlib.sha256()\\n    with Path(path).open(\'rb\') as f:\\n        for part in iter(lambda: f.read(1024 * 1024), b\'\'):\\n            h.update(part)\\n    return h.hexdigest()\\n\\n\\ndef read_json(path):\\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\\n\\n\\ndef write_json(path, value):\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + \'\\\\n\', encoding=\'utf-8\')\\n    temporary.replace(path)\\n\\n\\ndef _csv_text(value):\\n    text = str(value)\\n    # CSV is often opened in Excel. Model/source text must stay text.\\n    return \\"\'\\" + text if text.lstrip().startswith((\'=\', \'+\', \'-\', \'@\')) else text\\n\\n\\ndef _payload(result):\\n    claims = [{k: c[k] for k in (\'doc_id\', \'category\', \'text\', \'citations\')}\\n              for c in result.get(\'claims\', [])]\\n    for c in claims:\\n        c[\'citations\'] = [{k: cit[k] for k in (\'chunk_id\', \'quote\')} for cit in c[\'citations\']]\\n    missing = list(result.get(\'missing_information\', []))\\n    return {\'status\': result.get(\'status\'), \'claims\': claims,\\n            \'missing_information\': missing}\\n\\n\\ndef technical_check(result, sample, env, plan, request_fingerprints):\\n    \\"\\"\\"Revalidate exact quotes and field display, not just saved valid flags.\\"\\"\\"\\n    from rfp_phase5.fields import FIELDS, parse_field_claim\\n    checks = {}\\n    expected = list(plan[\'source_chunk_ids\'])\\n    batches = result.get(\'batch_outcomes\') or []\\n    checks[\'document_scope\'] = result.get(\'doc_id\') == sample[\'doc_id\']\\n    catalog = {r[\'doc_id\']: r for r in env[\'index\'].get_catalog()}\\n    checks[\'source_version\'] = catalog.get(sample[\'doc_id\'], {}).get(\'sha256\') == sample[\'source_sha256\']\\n    checks[\'plan_matches\'] = result.get(\'plan_fingerprint\') == plan[\'plan_fingerprint\']\\n    checks[\'all_source_chunks_listed\'] = bool(expected) and result.get(\'source_chunk_ids\') == expected\\n    checks[\'all_chunks_validated_once\'] = (bool(expected) and result.get(\'processed_chunk_ids\') == expected\\n        and len(expected) == len(set(expected)) and result.get(\'validated_chunk_coverage\') == 1.0\\n        and result.get(\'input_chunk_coverage\') == 1.0)\\n    checks[\'execution_completed\'] = result.get(\'execution_status\') == \'completed\'\\n    checks[\'answer_present\'] = result.get(\'answer_generated\') is True and bool(result.get(\'claims\'))\\n    checks[\'answer_status_usable\'] = result.get(\'status\') in {\'answered\', \'partial\'}\\n    checks[\'no_batch_errors\'] = not result.get(\'errors\') and not result.get(\'field_contract_failures\')\\n    checks[\'live_model_origin\'] = bool(batches) and all(b.get(\'valid\') is True\\n        and b.get(\'execution_mode\') == \'live_openai\' and b.get(\'live_api\') is True for b in batches)\\n    checks[\'batch_order_preserved\'] = ([cid for b in batches for cid in b.get(\'chunk_ids\', [])] == expected\\n        and [b.get(\'batch_id\') for b in batches] == [b[\'batch_id\'] for b in plan[\'batches\']])\\n    checks[\'exact_model_requests_match\'] = [b.get(\'request_fingerprint\') for b in batches] == list(request_fingerprints)\\n    checks[\'task_matches\'] = result.get(\'task\') == plan.get(\'task\')\\n    validation_errors = []\\n    try:\\n        chunks = env[\'index\'].get_document_chunks(sample[\'doc_id\'])\\n        validation = env[\'registry\'].validate_answer(_payload(copy.deepcopy(result)), chunks,\\n                                                       allowed_doc_ids=[sample[\'doc_id\']])\\n        validation_errors = validation.get(\'errors\', [])\\n        checks[\'quotes_revalidated\'] = validation.get(\'valid\') is True and bool(validation.get(\'claims\'))\\n        # The displayed fields must be a faithful view of the validated raw claims.\\n        derived = {f[\'id\']: [] for f in FIELDS}\\n        for claim in validation.get(\'claims\', []):\\n            parsed = parse_field_claim(claim)\\n            derived[parsed[\'field_id\']].append((parsed[\'text\'], parsed[\'mode\'],\\n                [(c[\'chunk_id\'], c[\'quote\']) for c in claim[\'citations\']]))\\n        fields = result.get(\'fields\') or []\\n        checks[\'eleven_fields_in_order\'] = [f.get(\'field_id\') for f in fields] == [f[\'id\'] for f in FIELDS]\\n        checks[\'field_display_matches_validated_claims\'] = checks[\'eleven_fields_in_order\'] and all(\\n            f.get(\'label\') == spec[\'label\'] and\\n            [(c.get(\'text\'), c.get(\'mode\'), [(q.get(\'chunk_id\'), q.get(\'quote\')) for q in c.get(\'citations\', [])])\\n             for c in f.get(\'claims\', [])] == derived[spec[\'id\']]\\n            for f, spec in zip(fields, FIELDS))\\n        checks[\'no_unprocessed_fields\'] = len(fields) == 11 and all(f.get(\'state\') in {\\n            \'evidence_found\', \'notice_reference\', \'not_found_in_processed_input\', \'conflict_candidates\'} for f in fields)\\n        from rfp_phase5.summary import FieldSummarizer\\n        normalized = FieldSummarizer._field_result(None, copy.deepcopy(result))\\n        checks[\'field_states_match_claims\'] = [(f.get(\'field_id\'), f.get(\'state\')) for f in fields] == [\\n            (f[\'field_id\'], f[\'state\']) for f in normalized[\'fields\']]\\n    except (ValueError, TypeError, KeyError, AttributeError):\\n        checks[\'quotes_revalidated\'] = False\\n        checks[\'field_display_matches_validated_claims\'] = False\\n    return {\'passed\': all(checks.values()), \'checks\': checks,\\n            \'failed_checks\': [k for k, v in checks.items() if not v],\\n            \'validation_errors\': validation_errors,\\n            \'semantic_quality_verified\': False,\\n            \'partial_status_is_not_automatic_failure_or_pass\': True}\\n\\n\\ndef make_review_rows(result, sample, references, source_audit=None):\\n    fields = {f[\'field_id\']: f for f in result.get(\'fields\', [])}\\n    rows = []\\n    for ref in [r for r in references[\'rows\'] if r[\'sample_id\'] == sample[\'sample_id\']]:\\n        field = fields.get(ref[\'field_id\'], {})\\n        citations = []\\n        for claim in field.get(\'claims\', []):\\n            for c in claim.get(\'citations\', []):\\n                text = str(c.get(\'locator\', \'\')) + \' | \' + str(c.get(\'quote\', \'\'))\\n                if text not in citations:\\n                    citations.append(text)\\n        candidates = [c for c in (source_audit or {}).get(\'candidates\', []) if c.get(\'field_id\') == ref[\'field_id\']]\\n        hints = \'\\\\n\'.join(\' / \'.join(dict.fromkeys(str(q.get(\'locator\', \'\')) for q in c.get(\'citations\', [])))\\n            + \' | \' + str(c.get(\'source_quote\', \'\')) for c in candidates)\\n        omitted = []\\n        for batch in result.get(\'batch_outcomes\', []):\\n            for item in batch.get(\'citation_diagnostics\', []):\\n                if item.get(\'claim_field_id\') == ref[\'field_id\'] and item.get(\'requires_conditions_review\'):\\n                    for gap in item.get(\'omitted_source_gaps\', []):\\n                        omitted.append(\'모델 인용에서 생략된 원문(답변에 자동 추가 안 함): \'\\n                            + str(item.get(\'chunk_id\', \'\')) + \' | \' + str(gap.get(\'text\', gap.get(\'preview\', \'\'))))\\n        hints = \'\\\\n\'.join([hints, *omitted]).strip()\\n        values = [ref[\'field_id\'], ref[\'label\'], ref[\'expected\'], ref[\'conditions\'],\\n            \'\\\\n\'.join(c.get(\'text\', \'\') for c in field.get(\'claims\', [])) or \'검증된 근거를 확보하지 못함\',\\n            field.get(\'state\', \'not_processed\'), ref[\'source_reference\'], \'\\\\n\'.join(citations), hints]\\n        row = {k: _csv_text(v) for k, v in zip(BASE_COLUMNS, values)}\\n        row.update({k: \'\' for k in (*JUDGEMENTS, \'메모\')})\\n        rows.append(row)\\n    if len(rows) != 11 or len({r[\'항목ID\'] for r in rows}) != 11:\\n        raise ValueError(\'검토 기준은 문서별 11개 항목이어야 합니다.\')\\n    return rows\\n\\n\\ndef write_review(path, rows):\\n    \\"\\"\\"Never replace completed or partially completed reviewer input.\\"\\"\\"\\n    path = Path(path)\\n    if path.exists():\\n        return\\n    with path.open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as f:\\n        writer = csv.DictWriter(f, fieldnames=[*BASE_COLUMNS, *JUDGEMENTS, \'메모\'])\\n        writer.writeheader()\\n        writer.writerows(rows)\\n\\n\\ndef assess_csv(path, expected_rows, reviewer):\\n    if not Path(path).is_file():\\n        return {\'passed\': False, \'status\': \'review_missing\', \'errors\': [], \'reviewed\': 0, \'total\': 33}\\n    with Path(path).open(encoding=\'utf-8-sig\', newline=\'\') as f:\\n        reader = csv.DictReader(f)\\n        rows = list(reader)\\n        headers = reader.fieldnames\\n    errors, pending, rejected, count = [], [], [], 0\\n    if headers != [*BASE_COLUMNS, *JUDGEMENTS, \'메모\']:\\n        errors.append(\'review_columns_changed\')\\n    if len(rows) != len(expected_rows):\\n        errors.append(\'review_row_count_changed\')\\n    for row, expected in zip(rows, expected_rows):\\n        if any(row.get(k) != expected[k] for k in BASE_COLUMNS):\\n            errors.append(\'answer_or_reference_cells_changed:\' + expected[\'항목ID\'])\\n        for key in JUDGEMENTS:\\n            value = str(row.get(key, \'\')).strip().lower()\\n            item = {\'field_id\': expected[\'항목ID\'], \'judgement\': key}\\n            if value in {\'통과\', \'true\'}:\\n                count += 1\\n            elif value in {\'실패\', \'false\'}:\\n                rejected.append(item)\\n            elif value in {\'\', \'미검토\'}:\\n                pending.append(item)\\n            else:\\n                errors.append(\'invalid_judgement:\' + expected[\'항목ID\'] + \':\' + key)\\n    if not isinstance(reviewer, str) or not reviewer.strip():\\n        pending.append({\'field_id\': None, \'judgement\': \'검토자 이름\'})\\n    status = (\'invalid_review\' if errors else \'review_failed\' if rejected else\\n              \'review_pending\' if pending else \'passed_human_review\')\\n    return {\'passed\': status == \'passed_human_review\', \'status\': status,\\n            \'errors\': sorted(set(errors)), \'pending\': pending, \'failed\': rejected,\\n            \'reviewed\': count + len(rejected), \'positive\': count, \'total\': 33,\\n            \'reviewer\': reviewer.strip(), \'checked_at_utc\': datetime.now(timezone.utc).isoformat()}\\n\\n\\ndef case_verdict(technical, human):\\n    return {\'accepted\': technical.get(\'passed\') is True and human.get(\'passed\') is True,\\n            \'technical_passed\': technical.get(\'passed\') is True,\\n            \'human_status\': human.get(\'status\'), \'technical_failures\': technical.get(\'failed_checks\', []),\\n            \'status\': (\'passed_sample_case\' if technical.get(\'passed\') and human.get(\'passed\') else\\n                       \'technical_failed\' if not technical.get(\'passed\') else human.get(\'status\', \'review_pending\'))}\\n\\n\\nclass ValidationSuite:\\n    def __init__(self, env, references, suite_id=\'phase5_final_v1\', *, per_doc_ceiling=900000,\\n                 suite_calls=75, suite_bytes=2400000):\\n        from rfp_phase5.runtime import make_components\\n        if not re.fullmatch(r\'[A-Za-z0-9_-]{1,64}\', suite_id):\\n            raise ValueError(\'SUITE_ID는 영문·숫자·밑줄·하이픈 1~64자로 지정하세요.\')\\n        if any(type(v) is not int or v < 1 for v in (per_doc_ceiling, suite_calls, suite_bytes)):\\n            raise ValueError(\'실행 상한은 양의 정수여야 합니다.\')\\n        self.env, self.references = env, copy.deepcopy(references)\\n        self.root = Path(env[\'root\'])\\n        self.samples = references[\'samples\']\\n        if len(self.samples) != 7 or len({s[\'sample_id\'] for s in self.samples}) != 7:\\n            raise ValueError(\'검증 문서는 개발 5건·최종 2건으로 고정합니다.\')\\n        if [s[\'split\'] for s in self.samples] != [\'dev\'] * 5 + [\'final\'] * 2:\\n            raise ValueError(\'검증 문서의 개발·최종 구분이 다릅니다.\')\\n        self.folder = self.root / \'phase5_outputs/final_validation\' / suite_id\\n        self.suite_calls, self.suite_bytes = suite_calls, suite_bytes\\n        self.plans, self.configs, self.request_fps = {}, {}, {}\\n        cat = {r[\'doc_id\']: r for r in env[\'index\'].get_catalog()}\\n        for sample in self.samples:\\n            sid = sample[\'sample_id\']\\n            if cat.get(sample[\'doc_id\'], {}).get(\'sha256\') != sample[\'source_sha256\']:\\n                raise ValueError(\'표본 문서 버전 불일치: \' + sid)\\n            component = make_components(env, doc_ids=[sample[\'doc_id\']], enabled=False)\\n            plan = component[\'summarizer\'].plan(sample[\'doc_id\'])\\n            if plan[\'chunk_count\'] != sample[\'expected_chunks\']:\\n                raise ValueError(\'표본 청크 개수가 기준표와 다릅니다: \' + sid)\\n            if not plan[\'can_execute\']:\\n                raise ValueError(\'문서 전체 입력 계획을 실행할 수 없습니다: \' + sid + \':\' + \',\'.join(plan[\'block_reasons\']))\\n            if plan[\'total_request_bytes\'] > per_doc_ceiling:\\n                raise ValueError(\'문서 전체 입력이 문서별 상한을 넘었습니다: \' + sid)\\n            local_env = dict(env)\\n            config = copy.deepcopy(env[\'phase5_config\'])\\n            current_limit = config[\'generation\'][\'max_total_input_bytes\']\\n            if plan[\'total_request_bytes\'] > current_limit:\\n                needed = ((plan[\'total_request_bytes\'] + 9999) // 10000) * 10000\\n                if needed > per_doc_ceiling:\\n                    raise ValueError(\'문서 전체 입력이 문서별 상한을 넘었습니다: \' + sid)\\n                config[\'generation\'][\'max_total_input_bytes\'] = needed\\n            local_env[\'phase5_config\'] = config\\n            # enabled=True creates no SDK client and does not make a request.\\n            prepared = make_components(local_env, doc_ids=[sample[\'doc_id\']], enabled=True)\\n            live_plan = prepared[\'summarizer\'].plan(sample[\'doc_id\'])\\n            self.plans[sid], self.configs[sid] = live_plan, config\\n            from rfp_phase5.fields import FIELD_SCHEMA, FIELD_QUESTION\\n            from rfp_phase5.summary import TASK\\n            self.request_fps[sid] = [prepared[\'llm\']._prepare(TASK, FIELD_QUESTION, b[\'chunks\'], schema=FIELD_SCHEMA)[\'fingerprint\']\\n                for b in live_plan[\'batches\']]\\n        self.frozen = {\'version\': VERSION, \'samples\': self.samples,\\n            \'reference_fingerprint\': digest(references),\\n            \'code_sha256\': {p: file_hash(self.root / p) for p in CORE_FILES},\\n            \'runner_sha256\': file_hash(__file__),\\n            \'index_fingerprint\': env[\'index\'].get_metadata()[\'index_fingerprint\'],\\n            \'dependencies\': env[\'environment\'][\'current_dependencies\'],\\n            \'config_by_sample\': self.configs,\\n            \'plan_fingerprints\': {s: p[\'plan_fingerprint\'] for s, p in self.plans.items()},\\n            \'suite_calls_cap\': suite_calls, \'suite_input_bytes_cap\': suite_bytes,\\n            \'per_doc_input_ceiling\': per_doc_ceiling}\\n        self.fingerprint = digest(self.frozen)\\n        self.folder.mkdir(parents=True, exist_ok=True)\\n        freeze_path = self.folder / \'frozen.json\'\\n        if freeze_path.exists():\\n            if read_json(freeze_path) != self.frozen:\\n                raise ValueError(\'코드·설정·원문·기준표가 변경됐습니다. 새 SUITE_ID로 개발 평가부터 시작하세요.\')\\n        else:\\n            write_json(freeze_path, self.frozen)\\n        write_json(self.folder / \'plan_summary.json\', self.plan_summary())\\n\\n    def assert_frozen(self):\\n        if read_json(self.folder / \'frozen.json\') != self.frozen:\\n            raise ValueError(\'고정 실행 기록이 변경됐습니다.\')\\n        if digest(self.references) != self.frozen[\'reference_fingerprint\'] or self.configs != self.frozen[\'config_by_sample\']:\\n            raise ValueError(\'평가 중 기준표 또는 설정이 변경됐습니다.\')\\n        current = {p: file_hash(self.root / p) for p in CORE_FILES}\\n        if current != self.frozen[\'code_sha256\'] or file_hash(__file__) != self.frozen[\'runner_sha256\']:\\n            raise ValueError(\'평가 중 코드가 변경됐습니다. 커널 재시작 후 새 SUITE_ID를 사용하세요.\')\\n        from rfp_phase4.runtime import environment_report\\n        if environment_report(self.root)[\'current_dependencies\'] != self.frozen[\'dependencies\']:\\n            raise ValueError(\'평가 중 의존성 버전이 변경됐습니다.\')\\n        from local_rag.io import verify_saved_run\\n        verify_saved_run(self.root)\\n\\n    def plan_summary(self):\\n        return [{\'표본\': s[\'sample_id\'], \'구분\': s[\'split\'], \'문서명\': s[\'file_name\'],\\n            \'청크\': self.plans[s[\'sample_id\']][\'chunk_count\'],\\n            \'캐시없을때_최대요청\': self.plans[s[\'sample_id\']][\'map_calls\'],\\n            \'입력byte\': self.plans[s[\'sample_id\']][\'total_request_bytes\'],\\n            \'문서별_입력상한\': self.configs[s[\'sample_id\']][\'generation\'][\'max_total_input_bytes\']}\\n            for s in self.samples]\\n\\n    def _component(self, sample, enabled):\\n        from rfp_phase5.runtime import make_components\\n        env = dict(self.env)\\n        env[\'phase5_config\'] = copy.deepcopy(self.configs[sample[\'sample_id\']])\\n        return make_components(env, doc_ids=[sample[\'doc_id\']], enabled=enabled)\\n\\n    def _check(self, result, sample):\\n        sid = sample[\'sample_id\']\\n        return technical_check(result, sample, self.env, self.plans[sid], self.request_fps[sid])\\n\\n    def record_result(self, sample, result, *, origin, session_usage=None):\\n        from rfp_phase5.review import answer_fingerprint\\n        from rfp_phase5.quality import audit_result\\n        sid = sample[\'sample_id\']\\n        case = self.folder / sid\\n        if (case / \'result.json\').exists():\\n            raise FileExistsError(\'기존 결과는 덮어쓰거나 자동 재시도하지 않습니다: \' + sid)\\n        if case.exists() and any(p.name != \'started.json\' for p in case.iterdir()):\\n            raise RuntimeError(\'기록이 중단된 폴더입니다. 파일을 보존하고 새 SUITE_ID를 사용하세요: \' + sid)\\n        case.mkdir(parents=True, exist_ok=True)\\n        write_json(case / \'result.json\', result)\\n        usage = dict(session_usage or {})\\n        write_json(case / \'origin.json\', {\'origin\': origin, \'suite_fingerprint\': self.fingerprint,\\n            \'answer_fingerprint\': answer_fingerprint(result),\\n            \'usage_in_this_suite\': usage, \'created_at_utc\': datetime.now(timezone.utc).isoformat()})\\n        (case / \'answer.md\').write_text(str(result.get(\'markdown\', \'생성 답변 없음\')) + \'\\\\n\', encoding=\'utf-8\')\\n        write_json(case / \'surface_audit.json\', audit_result(result))\\n        self.assess_case(sample, reviewer=\'\')\\n\\n    def assess_case(self, sample, reviewer):\\n        from rfp_phase5.review import answer_fingerprint\\n        case = self.folder / sample[\'sample_id\']\\n        if not (case / \'result.json\').is_file():\\n            return {\'sample_id\': sample[\'sample_id\'], \'split\': sample[\'split\'], \'accepted\': False,\\n                    \'status\': \'not_generated\', \'technical_passed\': False}\\n        result = read_json(case / \'result.json\')\\n        origin = read_json(case / \'origin.json\')\\n        if origin[\'suite_fingerprint\'] != self.fingerprint or origin[\'answer_fingerprint\'] != answer_fingerprint(result):\\n            raise ValueError(\'평가 결과 또는 실행 기록이 변경됐습니다: \' + sample[\'sample_id\'])\\n        technical = self._check(result, sample)\\n        write_json(case / \'technical_check.json\', technical)\\n        source_audit = {}\\n        if technical[\'passed\']:\\n            from rfp_phase5.source_audit import audit_summary_sources\\n            source_audit = audit_summary_sources(self.env[\'index\'], self.env[\'registry\'], result)\\n            write_json(case / \'source_audit.json\', source_audit)\\n            expected_rows = make_review_rows(result, sample, self.references, source_audit)\\n            write_review(case / \'comparison.csv\', expected_rows)\\n            human = assess_csv(case / \'comparison.csv\', expected_rows, reviewer)\\n        else:\\n            human = {\'passed\': False, \'status\': \'technical_failed_before_review\', \'reviewed\': 0, \'total\': 33}\\n        write_json(case / \'human_review_assessment.json\', human)\\n        verdict = {**case_verdict(technical, human), \'sample_id\': sample[\'sample_id\'], \'split\': sample[\'split\'],\\n            \'model_status\': result.get(\'status\'), \'execution_status\': result.get(\'execution_status\'),\\n            \'validated_chunk_coverage\': result.get(\'validated_chunk_coverage\'),\\n            \'reviewed_judgements\': human.get(\'reviewed\', 0), \'judgements\': 33,\\n            \'source_audit_candidates\': len(source_audit.get(\'candidates\', [])),\\n            \'citation_gap_review_count\': sum(len(d.get(\'omitted_source_gaps\', [])) for b in result.get(\'batch_outcomes\', [])\\n                for d in b.get(\'citation_diagnostics\', []) if d.get(\'requires_conditions_review\')),\\n            \'result_path\': str(case / \'result.json\'), \'review_path\': str(case / \'comparison.csv\')}\\n        write_json(case / \'acceptance.json\', verdict)\\n        return verdict\\n\\n    def import_existing(self, split=\'dev\'):\\n        \\"\\"\\"Reuse only fully revalidated results of the identical current requests.\\"\\"\\"\\n        self.assert_frozen()\\n        paths = list((self.root / \'phase5_outputs/runs\').glob(\'*/result.json\'))\\n        for previous in (self.root / \'phase5_outputs/final_validation\').glob(\'*/\'):\\n            if previous.resolve() != self.folder.resolve():\\n                paths.extend(previous.glob(\'*/result.json\'))\\n        paths = sorted(set(paths), reverse=True)\\n        imported = []\\n        for sample in [s for s in self.samples if s[\'split\'] == split]:\\n            if (self.folder / sample[\'sample_id\'] / \'result.json\').exists():\\n                continue\\n            for path in paths:\\n                try:\\n                    result = read_json(path)\\n                    if result.get(\'doc_id\') != sample[\'doc_id\'] or not self._check(result, sample)[\'passed\']:\\n                        continue\\n                except (ValueError, TypeError, KeyError):\\n                    continue\\n                self.record_result(sample, result, origin=\'reused_existing_exact_requests:\' + str(path))\\n                imported.append(sample[\'sample_id\'])\\n                break\\n        return imported\\n\\n    def used_budget(self):\\n        calls, size = 0, 0\\n        for sample in self.samples:\\n            path = self.folder / sample[\'sample_id\'] / \'origin.json\'\\n            if path.is_file():\\n                usage = read_json(path).get(\'usage_in_this_suite\', {})\\n                calls += usage.get(\'api_calls\', 0)\\n                size += usage.get(\'attempted_input_bytes\', 0)\\n        return {\'api_calls\': calls, \'attempted_input_bytes\': size}\\n\\n    def run_stage(self, split, *, enabled=False, reviewer=\'\'):\\n        if split not in {\'dev\', \'final\'} or type(enabled) is not bool:\\n            raise ValueError(\'실행 구분 또는 API 실행 옵션이 올바르지 않습니다.\')\\n        self.assert_frozen()\\n        current = self.report(reviewer)\\n        if split == \'final\' and not current[\'dev_passed\']:\\n            return {\'status\': \'blocked_until_dev_review_passes\', \'api_calls\': 0,\\n                    \'instruction\': \'개발 5건의 comparison.csv를 원문과 대조하고 검토자 이름을 입력하세요.\'}\\n        self.import_existing(split)\\n        existing_failures = [c[\'sample_id\'] for c in current.get(\'cases\', []) if c[\'split\'] == split\\n            and c.get(\'status\') == \'technical_failed\']\\n        if existing_failures:\\n            return {\'status\': \'blocked_by_existing_technical_failure\', \'api_calls\': 0,\\n                    \'samples\': existing_failures, \'automatic_retry\': False}\\n        selected = [s for s in self.samples if s[\'split\'] == split and\\n                    not (self.folder / s[\'sample_id\'] / \'result.json\').exists()]\\n        interrupted = [s[\'sample_id\'] for s in selected if (self.folder / s[\'sample_id\'] / \'started.json\').exists()]\\n        if interrupted:\\n            return {\'status\': \'interrupted_attempt_requires_review\', \'api_calls\': 0,\\n                    \'samples\': interrupted, \'automatic_retry\': False}\\n        if not enabled:\\n            return {\'status\': \'dry_run\', \'api_calls\': 0, \'missing_samples\': [s[\'sample_id\'] for s in selected],\\n                    \'instruction\': \'API 실행 옵션을 켜면 아직 결과가 없는 문서만 실행합니다.\'}\\n        # Preflight every selected document before the first API request.\\n        prepared = []\\n        for sample in selected:\\n            component = self._component(sample, enabled=True)\\n            p = self.plans[sample[\'sample_id\']]\\n            requests = [component[\'summarizer\']._inner._request(p[\'question\'], b[\'chunks\']) for b in p[\'batches\']]\\n            reservation = component[\'summarizer\']._adapter.preflight_requests(requests)\\n            prepared.append((sample, component, reservation))\\n        used = self.used_budget()\\n        additional_calls = sum(p[2][\'uncached_requests\'] for p in prepared)\\n        additional_bytes = sum(p[2][\'input_bytes_to_reserve\'] for p in prepared)\\n        if used[\'api_calls\'] + additional_calls > self.suite_calls or used[\'attempted_input_bytes\'] + additional_bytes > self.suite_bytes:\\n            raise ValueError(\'검증 전체의 호출·입력 상한을 넘습니다. API 호출 없이 중단했습니다.\')\\n        for sample, component, reservation in prepared:\\n            self.assert_frozen()\\n            print(\'요약 실행:\', sample[\'sample_id\'], sample[\'file_name\'], flush=True)\\n            write_json(self.folder / sample[\'sample_id\'] / \'started.json\', {\\n                \'suite_fingerprint\': self.fingerprint, \'started_at_utc\': datetime.now(timezone.utc).isoformat(),\\n                \'scope\': \'완료 기록이 없으면 사용량을 확정할 수 없습니다. 자동 재시도하지 않습니다.\'})\\n            try:\\n                result = component[\'summarizer\'].summarize(sample[\'doc_id\'])\\n            except Exception as error:\\n                from rfp_phase4.llm import safe_llm_diagnostic\\n                result = {\'doc_id\': sample[\'doc_id\'], \'status\': \'error\', \'execution_status\': \'incomplete\',\\n                    \'answer_generated\': False, \'claims\': [], \'fields\': [],\\n                    \'errors\': [\'unexpected_runner_exception:\' + type(error).__name__],\\n                    \'error_diagnostics\': [safe_llm_diagnostic(error)],\\n                    \'usage\': {}, \'markdown\': \'요약을 완료하지 못했습니다. 진단 기록을 확인하세요.\'}\\n            self.record_result(sample, result, origin=\'current_suite_generation\',\\n                               session_usage=component[\'llm\'].usage_report())\\n            technical = self._check(result, sample)\\n            print(\'실행:\', result.get(\'execution_status\'), \'/ 출처·전체 입력:\', technical[\'passed\'], flush=True)\\n            if not technical[\'passed\']:\\n                return {\'status\': \'stopped_after_technical_failure\', \'sample_id\': sample[\'sample_id\'],\\n                        \'errors\': result.get(\'errors\'), \'error_detail\': result.get(\'error_detail\'),\\n                        \'error_diagnostics\': result.get(\'error_diagnostics\'),\\n                        \'failed_checks\': technical[\'failed_checks\'], \'automatic_retry\': False}\\n        return {\'status\': \'stage_finished\', \'samples_generated\': [p[0][\'sample_id\'] for p in prepared],\\n                \'used_budget\': self.used_budget(), \'human_review_pending\': True}\\n\\n    def report(self, reviewer=\'\'):\\n        cases = [self.assess_case(s, reviewer) for s in self.samples]\\n        dev_passed = all(c[\'accepted\'] for c in cases if c[\'split\'] == \'dev\')\\n        final_passed = all(c[\'accepted\'] for c in cases if c[\'split\'] == \'final\')\\n        tests_path = self.folder / \'offline_tests.json\'\\n        offline = read_json(tests_path) if tests_path.is_file() else {}\\n        tests_passed = offline.get(\'passed\') is True and offline.get(\'suite_fingerprint\') == self.fingerprint\\n        passed = dev_passed and final_passed and tests_passed\\n        report = {\'version\': VERSION, \'suite_fingerprint\': self.fingerprint,\\n            \'status\': \'passed_sample_validation\' if passed else \'validation_pending_or_failed\',\\n            \'phase5_sample_validation_passed\': passed, \'dev_passed\': dev_passed,\\n            \'final_passed\': final_passed, \'offline_failure_checks_passed\': tests_passed,\\n            \'scope\': \'고정된 공개 RFP 7건·77항목의 요약 검증. 전체 99건·Q&A 전체 성능을 확정하지 않습니다.\',\\n            \'all_documents_quality_verified\': False, \'cases\': cases,\\n            \'budget_this_suite\': self.used_budget(), \'output_folder\': str(self.folder)}\\n        write_json(self.folder / \'final_report.json\', report)\\n        return report\\n\\n    def run_offline_tests(self):\\n        \\"\\"\\"Real rejection paths, with sockets blocked. Never calls a model service.\\"\\"\\"\\n        test_root = self.root / \'tests\'\\n        validation_tests = Path(__file__).parent / \'test_final_validation.py\'\\n        cache_tests = Path(__file__).parent / \'test_cache_revalidation.py\'\\n        selected = [\'test_phase5_summary.py\', \'test_phase5_llm_diagnostics.py\',\\n                    \'test_phase5_saved_diagnostics.py\', \'test_phase5_source_audit.py\',\\n                    \'test_phase4_citation_recovery.py\', \'test_phase4_citation_split.py\']\\n        program = \'\'\'import importlib.util,json,pathlib,sys,unittest\\nfrom unittest.mock import patch\\nroot=pathlib.Path(sys.argv[1]); paths=json.loads(sys.argv[2])\\nsys.path.insert(0,str(root));sys.path.insert(0,str(root/\'tests\'))\\nsuite=unittest.TestSuite()\\nfor i,p in enumerate(paths):\\n spec=importlib.util.spec_from_file_location(\'validation_case_\'+str(i),p)\\n module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)\\n suite.addTests(unittest.defaultTestLoader.loadTestsFromModule(module))\\nwith patch(\'socket.socket\',side_effect=AssertionError(\'network prohibited in offline tests\')):\\n result=unittest.TextTestRunner(verbosity=1).run(suite)\\nprint(\'VALIDATION_RESULT=\'+json.dumps({\'passed\':result.wasSuccessful(),\'tests_run\':result.testsRun,\'failures\':len(result.failures),\'errors\':len(result.errors),\'skipped\':len(result.skipped),\'api_calls\':0}))\\n\'\'\'\\n        paths = [str(test_root / n) for n in selected] + [str(validation_tests), str(cache_tests)]\\n        if not all(Path(p).is_file() for p in paths):\\n            raise FileNotFoundError(\'09·10번 패치의 테스트 파일 또는 최종검증 테스트가 없습니다.\')\\n        run = subprocess.run([sys.executable, \'-c\', program, str(self.root), json.dumps(paths)],\\n                             capture_output=True, text=True, timeout=90, cwd=self.root)\\n        lines = [l for l in run.stdout.splitlines() if l.startswith(\'VALIDATION_RESULT=\')]\\n        if not lines:\\n            raise RuntimeError(\'오프라인 테스트를 완료하지 못했습니다. 현재 커널과 09·10번 패치를 확인하세요.\')\\n        result = json.loads(lines[-1].split(\'=\', 1)[1])\\n        result[\'passed\'] = result[\'passed\'] and run.returncode == 0 and result[\'skipped\'] == 0\\n        result[\'suite_fingerprint\'] = self.fingerprint\\n        write_json(self.folder / \'offline_tests.json\', result)\\n        (self.folder / \'offline_tests.log\').write_text(run.stderr, encoding=\'utf-8\')\\n        return result\\n\\n    def export(self, reviewer=\'\'):\\n        self.assert_frozen()\\n        self.report(reviewer)\\n        target = self.folder.parent / (self.folder.name + \'_results.zip\')\\n        # Only this suite\'s explicit records; no .env, keys, response cache, or corpus.\\n        with zipfile.ZipFile(target, \'w\', compression=zipfile.ZIP_DEFLATED) as z:\\n            for p in sorted(self.folder.rglob(\'*\')):\\n                if p.is_file() and p.suffix in {\'.json\', \'.csv\', \'.md\', \'.log\'}:\\n                    z.write(p, self.folder.name + \'/\' + p.relative_to(self.folder).as_posix())\\n        return target\\n", "phase5_validation/test_final_validation.py": "\\"\\"\\"Acceptance regressions with synthetic source-backed doubles. API calls: zero.\\"\\"\\"\\nimport copy\\nimport csv\\nimport importlib.util\\nfrom pathlib import Path\\nimport tempfile\\nimport unittest\\nfrom types import SimpleNamespace\\n\\nspec = importlib.util.spec_from_file_location(\'final_validation_under_test\', Path(__file__).with_name(\'final_validation.py\'))\\nv = importlib.util.module_from_spec(spec)\\nspec.loader.exec_module(v)\\n\\nimport test_phase5_summary as fixture\\n\\n\\ndef case():\\n    summary, llm, chunks = fixture.FieldSummaryTests().make(count=3,\\n        llm=fixture.OfflineLLM(all_fields=True), config={\'max_batch_bytes\': 9000})\\n    result = summary.summarize(\'d1\')\\n    plan = summary.plan(\'d1\')\\n    for n, batch in enumerate(result[\'batch_outcomes\']):\\n        batch.update(execution_mode=\'live_openai\', live_api=True, request_fingerprint=str(n+1))\\n    summary.index.get_catalog = lambda: [{\'doc_id\': \'d1\', \'sha256\': \'a\'*64}]\\n    env = {\'index\': summary.index, \'registry\': summary.registry}\\n    sample = {\'sample_id\': \'D01\', \'doc_id\': \'d1\', \'source_sha256\': \'a\'*64}\\n    fps = [str(n+1) for n in range(len(result[\'batch_outcomes\']))]\\n    refs = {\'rows\': [{\'sample_id\': \'D01\', \'field_id\': f[\'id\'], \'label\': f[\'label\'],\\n        \'expected\': \'원문 대조용 테스트 기준\', \'conditions\': \'조건 보존 검토\', \'source_reference\': \'PDF p.1\'}\\n        for f in fixture.FIELDS]}\\n    return result, sample, env, plan, fps, refs\\n\\n\\nclass FinalValidationTests(unittest.TestCase):\\n    def check(self, result, sample, env, plan, fps):\\n        return v.technical_check(result, sample, env, plan, fps)\\n\\n    def test_complete_multibatch_sources_pass_technical_only(self):\\n        r, s, e, p, fps, _ = case()\\n        self.assertGreater(p[\'map_calls\'], 1)\\n        check = self.check(r, s, e, p, fps)\\n        self.assertTrue(check[\'passed\'], check[\'failed_checks\'])\\n        self.assertFalse(check[\'semantic_quality_verified\'])\\n        self.assertFalse(v.case_verdict(check, {\'passed\': False, \'status\': \'review_pending\'})[\'accepted\'])\\n\\n    def test_partial_status_requires_complete_processing_and_human_review(self):\\n        r, s, e, p, fps, _ = case()\\n        r[\'status\'] = \'partial\'\\n        r[\'missing_information\'] = [\'제출기한은 공고문 별도 확인\']\\n        check = self.check(r, s, e, p, fps)\\n        self.assertTrue(check[\'passed\'], check[\'failed_checks\'])\\n        self.assertTrue(v.case_verdict(check, {\'passed\': True, \'status\': \'passed_human_review\'})[\'accepted\'])\\n        r[\'processed_chunk_ids\'] = r[\'processed_chunk_ids\'][:-1]\\n        self.assertFalse(self.check(r, s, e, p, fps)[\'passed\'])\\n\\n    def test_fabricated_quote_rejected_despite_saved_valid_flag(self):\\n        r, s, e, p, fps, _ = case()\\n        r[\'claims\'][0][\'citations\'][0][\'quote\'] = \'존재하지 않는 예산 999조원\'\\n        r[\'valid\'] = True\\n        self.assertFalse(self.check(r, s, e, p, fps)[\'passed\'])\\n\\n    def test_display_can_not_diverge_from_validated_claims(self):\\n        r, s, e, p, fps, _ = case()\\n        r[\'fields\'][0][\'claims\'][0][\'text\'] = \'모델 출력에 없는 사업명\'\\n        self.assertIn(\'field_display_matches_validated_claims\', self.check(r, s, e, p, fps)[\'failed_checks\'])\\n\\n    def test_field_state_can_not_be_changed_to_hide_uncertainty(self):\\n        r, s, e, p, fps, _ = case()\\n        r[\'fields\'][0][\'state\'] = \'not_found_in_processed_input\'\\n        self.assertIn(\'field_states_match_claims\', self.check(r, s, e, p, fps)[\'failed_checks\'])\\n\\n    def test_wrong_scope_source_and_request_all_rejected(self):\\n        for mode in [\'scope\', \'source\', \'request\', \'mock\', \'missing_field\', \'reorder\']:\\n            r, s, e, p, fps, _ = case()\\n            if mode == \'scope\': r[\'doc_id\'] = \'other\'\\n            if mode == \'source\': s[\'source_sha256\'] = \'b\'*64\\n            if mode == \'request\': fps[0] = \'different-config-or-prompt\'\\n            if mode == \'mock\': r[\'batch_outcomes\'][0][\'live_api\'] = False\\n            if mode == \'missing_field\': r[\'fields\'].pop()\\n            if mode == \'reorder\': r[\'batch_outcomes\'].reverse()\\n            with self.subTest(mode=mode): self.assertFalse(self.check(r, s, e, p, fps)[\'passed\'])\\n\\n    def test_duplicate_processing_ids_do_not_count_as_full_coverage(self):\\n        r, s, e, p, fps, _ = case()\\n        r[\'processed_chunk_ids\'] += r[\'processed_chunk_ids\']\\n        self.assertFalse(self.check(r, s, e, p, fps)[\'passed\'])\\n\\n    def test_cached_live_origin_allowed_without_new_paid_call(self):\\n        r, s, e, p, fps, _ = case()\\n        for batch in r[\'batch_outcomes\']:\\n            batch.update(cache_hit=True, api_called=False)\\n        self.assertTrue(self.check(r, s, e, p, fps)[\'passed\'])\\n\\n    def review_fixture(self, folder):\\n        r, s, e, p, fps, refs = case()\\n        rows = v.make_review_rows(r, s, refs)\\n        path = Path(folder) / \'comparison.csv\'\\n        v.write_review(path, rows)\\n        return path, rows\\n\\n    def fill(self, path, rows, value=\'통과\'):\\n        output = copy.deepcopy(rows)\\n        for row in output:\\n            for key in v.JUDGEMENTS: row[key] = value\\n        path.unlink()\\n        v.write_review(path, output)\\n        return output\\n\\n    def test_review_is_blank_and_never_implicitly_positive(self):\\n        with tempfile.TemporaryDirectory() as d:\\n            path, rows = self.review_fixture(d)\\n            report = v.assess_csv(path, rows, \'검토자\')\\n            self.assertEqual(report[\'status\'], \'review_pending\')\\n            self.assertEqual(report[\'reviewed\'], 0)\\n\\n    def test_all_33_judgements_and_reviewer_required(self):\\n        with tempfile.TemporaryDirectory() as d:\\n            path, rows = self.review_fixture(d)\\n            self.fill(path, rows)\\n            self.assertFalse(v.assess_csv(path, rows, \'\')[\'passed\'])\\n            report = v.assess_csv(path, rows, \'원문 검토자\')\\n            self.assertTrue(report[\'passed\'])\\n            self.assertEqual(report[\'positive\'], 33)\\n\\n    def test_negative_and_invalid_judgements_block_acceptance(self):\\n        for value in [\'실패\', \'1\', \'=TRUE()\']:\\n            with self.subTest(value=value), tempfile.TemporaryDirectory() as d:\\n                path, rows = self.review_fixture(d)\\n                self.fill(path, rows, value)\\n                self.assertFalse(v.assess_csv(path, rows, \'검토자\')[\'passed\'])\\n\\n    def test_review_bound_to_unchanged_answer_and_reference(self):\\n        with tempfile.TemporaryDirectory() as d:\\n            path, rows = self.review_fixture(d)\\n            changed = self.fill(path, rows)\\n            changed[0][\'실제요약\'] = \'사용자가 표에서 바꾼 답변\'\\n            path.unlink(); v.write_review(path, changed)\\n            self.assertEqual(v.assess_csv(path, rows, \'검토자\')[\'status\'], \'invalid_review\')\\n\\n    def test_csv_input_is_preserved_on_rerun(self):\\n        with tempfile.TemporaryDirectory() as d:\\n            path, rows = self.review_fixture(d)\\n            self.fill(path, rows)\\n            before = path.read_bytes()\\n            v.write_review(path, rows)\\n            self.assertEqual(before, path.read_bytes())\\n\\n    def test_csv_formula_text_is_escaped(self):\\n        self.assertEqual(v._csv_text(\'=HYPERLINK(\\"bad\\")\'), \'\\\\\'=HYPERLINK(\\"bad\\")\')\\n        self.assertEqual(v._csv_text(\'650,000,000원\'), \'650,000,000원\')\\n\\n    def test_invalid_suite_id_rejected_before_any_components_or_api(self):\\n        with self.assertRaises(ValueError): v.ValidationSuite({}, {}, \'../other\')\\n\\n    def stub_suite(self, folder):\\n        obj = v.ValidationSuite.__new__(v.ValidationSuite)\\n        obj.folder = Path(folder)\\n        obj.samples = [{\'sample_id\': \'D01\', \'split\': \'dev\', \'doc_id\': \'d1\', \'file_name\': \'공개 문서\'}]\\n        obj.assert_frozen = lambda: None\\n        obj.report = lambda reviewer: {\'dev_passed\': False}\\n        obj.import_existing = lambda split: []\\n        obj.used_budget = lambda: {\'api_calls\': 0, \'attempted_input_bytes\': 0}\\n        obj.plans = {\'D01\': {\'question\': \'형식 검사\', \'batches\': []}}\\n        obj.suite_calls, obj.suite_bytes = 1, 100\\n        return obj\\n\\n    def test_final_stage_does_not_generate_before_dev_human_review(self):\\n        with tempfile.TemporaryDirectory() as d:\\n            obj = self.stub_suite(d)\\n            obj._component = lambda *a, **kw: self.fail(\'검토 전 API 준비도 하지 않아야 함\')\\n            self.assertEqual(obj.run_stage(\'final\', enabled=True)[\'status\'], \'blocked_until_dev_review_passes\')\\n\\n    def test_whole_suite_cap_blocks_before_generate(self):\\n        with tempfile.TemporaryDirectory() as d:\\n            obj = self.stub_suite(d)\\n            summary = SimpleNamespace(_adapter=SimpleNamespace(preflight_requests=lambda _: {\\n                \'uncached_requests\': 2, \'input_bytes_to_reserve\': 50}))\\n            summary.summarize = lambda *a: self.fail(\'상한을 넘으면 생성하면 안 됨\')\\n            obj._component = lambda *a, **kw: {\'summarizer\': summary}\\n            with self.assertRaises(ValueError): obj.run_stage(\'dev\', enabled=True)\\n\\n    def test_interrupted_attempt_blocks_automatic_retry(self):\\n        with tempfile.TemporaryDirectory() as d:\\n            obj = self.stub_suite(d)\\n            path = Path(d)/\'D01\';path.mkdir();(path/\'started.json\').write_text(\'{}\')\\n            obj._component = lambda *a, **kw: self.fail(\'중단된 요청의 자동 재시도 금지\')\\n            self.assertEqual(obj.run_stage(\'dev\', enabled=True)[\'status\'], \'interrupted_attempt_requires_review\')\\n\\n    def test_saved_result_is_not_generated_again_on_run_all(self):\\n        with tempfile.TemporaryDirectory() as d:\\n            obj = self.stub_suite(d)\\n            path = Path(d)/\'D01\';path.mkdir();(path/\'result.json\').write_text(\'{\\"status\\":\\"error\\"}\')\\n            obj._component = lambda *a, **kw: self.fail(\'실패도 명시적 새 실행 전까지 보존\')\\n            self.assertEqual(obj.run_stage(\'dev\', enabled=True)[\'samples_generated\'], [])\\n\\n    def test_saved_technical_failure_stops_remaining_document_calls(self):\\n        with tempfile.TemporaryDirectory() as d:\\n            obj = self.stub_suite(d)\\n            obj.report = lambda reviewer: {\'dev_passed\': False, \'cases\': [\\n                {\'sample_id\': \'D01\', \'split\': \'dev\', \'status\': \'technical_failed\'}]}\\n            obj._component = lambda *a, **kw: self.fail(\'기존 실패가 있으면 추가 요청 중단\')\\n            self.assertEqual(obj.run_stage(\'dev\', enabled=True)[\'status\'], \'blocked_by_existing_technical_failure\')\\n\\n    def test_accepted_split_keeps_omitted_conditions_for_review_without_adding_claims(self):\\n        from test_phase4_evidence import block, document, chunk\\n        from rfp_phase4.evidence import EvidenceRegistry\\n        from rfp_phase5.summary import FieldSummarizer\\n        a=\'사업예산은 650백만원이며 부가세를 포함합니다.\'\\n        b=\'단가계약의 대상인원은 실제 인원으로 정산합니다.\'\\n        c=\'가격제안서는 별도로 밀봉하여 제출해야 합니다.\'\\n        source=a+\'\\\\n공동수급은 허용되지 않습니다.\\\\n\'+b+\'\\\\n접수기한 이후 변경할 수 없습니다.\\\\n\'+c\\n        source_block=block(text=source);doc=document(source=source_block);ch=chunk(source_block)\\n        class SplitDouble(fixture.OfflineLLM):\\n            def generate(self,*args,**kwargs):\\n                r=super().generate(*args,**kwargs)\\n                r[\'payload\'][\'claims\'][0][\'citations\'][0][\'quote\']=a+\'\\\\n\'+b+\'\\\\n\'+c\\n                return r\\n        llm=SplitDouble()\\n        result=FieldSummarizer(fixture.Index([ch]),llm,EvidenceRegistry([doc],[ch])).summarize(\'d1\')\\n        self.assertEqual(len(llm.calls),1)\\n        self.assertEqual(len(result[\'claims\']),1)\\n        self.assertEqual(len(result[\'claims\'][0][\'citations\']),3)\\n        diagnostic=result[\'batch_outcomes\'][0][\'citation_diagnostics\'][0]\\n        self.assertEqual(diagnostic[\'claim_field_id\'],\'budget_vat\')\\n        self.assertEqual(len(diagnostic[\'omitted_source_gaps\']),2)\\n        _,sample,_,_,_,refs=case()\\n        rows=v.make_review_rows(result,sample,refs)\\n        budget=next(row for row in rows if row[\'항목ID\']==\'budget_vat\')\\n        self.assertIn(\'공동수급은 허용되지 않습니다\',budget[\'추가원문확인후보\'])\\n        self.assertNotIn(\'공동수급은 허용되지 않습니다\',budget[\'실제요약\'])\\n\\n\\nif __name__ == \'__main__\':\\n    unittest.main()\\n", "phase5_validation/references.json": "{\\n  \\"version\\": \\"phase5-final-references-v1\\",\\n  \\"samples\\": [\\n    {\\n      \\"sample_id\\": \\"D01\\",\\n      \\"split\\": \\"dev\\",\\n      \\"file_name\\": \\"인천공항시설관리(주)_2026_2026년도 귀속 연말정산 위탁운영 용역_과업내용서.hwp\\",\\n      \\"doc_id\\": \\"1Eq8gIdUnnLRbbEBiaU29wYji2YglNNuu\\",\\n      \\"source_sha256\\": \\"7a592ea538ed00d8dca697c8cab8fadd110ec2a15902f8235b80bb7d960027bc\\",\\n      \\"expected_chunks\\": 18,\\n      \\"planned_calls\\": 1\\n    },\\n    {\\n      \\"sample_id\\": \\"D02\\",\\n      \\"split\\": \\"dev\\",\\n      \\"file_name\\": \\"서울특별시_2026_2027년 글로벌 패션브랜드 육성 용역_제안요청서.hwpx\\",\\n      \\"doc_id\\": \\"1Uy4SDf7e_NFKK9CbjL9uVO4ta_MQdeTt\\",\\n      \\"source_sha256\\": \\"081719865dd99a38ba822b39869efcf75b37add88f8d07741c8f8bf141060484\\",\\n      \\"expected_chunks\\": 154,\\n      \\"planned_calls\\": 6\\n    },\\n    {\\n      \\"sample_id\\": \\"D03\\",\\n      \\"split\\": \\"dev\\",\\n      \\"file_name\\": \\"한국주택금융공사_2023_HF 인사이트 시스템 개발_제안요청서.hwp\\",\\n      \\"doc_id\\": \\"1G2at8Un7y8JIn5awuVxqV7OOKIj7zT4B\\",\\n      \\"source_sha256\\": \\"8c293ac2bb077438c0aaa974746d203a9196ec3eaa0d2c7971e5f6d3e4a28e28\\",\\n      \\"expected_chunks\\": 442,\\n      \\"planned_calls\\": 19\\n    },\\n    {\\n      \\"sample_id\\": \\"D04\\",\\n      \\"split\\": \\"dev\\",\\n      \\"file_name\\": \\"한국주택금융공사_2021_부동산 등기정보 시스템 구축_제안요청서.hwp\\",\\n      \\"doc_id\\": \\"1MxTCLhVvegbCCvqonlf95yO6pr4fYgaJ\\",\\n      \\"source_sha256\\": \\"a3159eeb228799678e146447a070ba3c6f609e9723667c1fe768c75cf76b325c\\",\\n      \\"expected_chunks\\": 202,\\n      \\"planned_calls\\": 11\\n    },\\n    {\\n      \\"sample_id\\": \\"D05\\",\\n      \\"split\\": \\"dev\\",\\n      \\"file_name\\": \\"한국관광공사_2026_AI 기반 관광 B2C 플랫폼 통합 ISP 수립_제안요청서.hwp\\",\\n      \\"doc_id\\": \\"161LuHCu_ZzUIDue5yGL_rRKMUe10FZ38\\",\\n      \\"source_sha256\\": \\"a0429ca4da02110586ecf5396ca89edf17c53dd26db9b26671f0802892791a8e\\",\\n      \\"expected_chunks\\": 245,\\n      \\"planned_calls\\": 13\\n    },\\n    {\\n      \\"sample_id\\": \\"R06\\",\\n      \\"split\\": \\"final\\",\\n      \\"file_name\\": \\"건설엔지니어링공제조합_2025_조합 전산시스템 고도화 사업(전면개편)_제안요청서.pdf\\",\\n      \\"doc_id\\": \\"19K21s_OFwsevxfq9PJZFMSkNJVt9SPM0\\",\\n      \\"source_sha256\\": \\"339f63a565c0a45e88f9f7efb59707ed743510e74575cbb4ae21358410cf5ead\\",\\n      \\"expected_chunks\\": 55,\\n      \\"planned_calls\\": 4\\n    },\\n    {\\n      \\"sample_id\\": \\"R07\\",\\n      \\"split\\": \\"final\\",\\n      \\"file_name\\": \\"한국정보통신진흥협회_2026_2026년 AI연구용 컴퓨팅 지원 프로젝트 GPU 자원 공급 및 유지보수_제안요청서.hwp\\",\\n      \\"doc_id\\": \\"1ZyM6Ipx-Zv-5E63Vb16dqhEy9VhsokLc\\",\\n      \\"source_sha256\\": \\"3cd93bd8e0f92667d7f0d681d7c64070f810f6ff0035f2ea564a7b56f19072c1\\",\\n      \\"expected_chunks\\": 51,\\n      \\"planned_calls\\": 3\\n    }\\n  ],\\n  \\"rows\\": [\\n    {\\n      \\"sample_id\\": \\"D01\\",\\n      \\"field_id\\": \\"project_name\\",\\n      \\"label\\": \\"사업명\\",\\n      \\"expected\\": \\"2026년도 귀속 연말정산 위탁운영 용역\\",\\n      \\"conditions\\": \\"사업연도 2026을 문서 작성일·실제 수행 종료연도와 구별한다.\\",\\n      \\"source_reference\\": \\"문단 10 외 1곳\\\\nD01-01-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D01\\",\\n      \\"field_id\\": \\"issuing_agency\\",\\n      \\"label\\": \\"발주기관\\",\\n      \\"expected\\": \\"[부분확인]\\\\n본문에 인천공항시설관리(주)가 등장한다. 발주기관 역할의 최종 확인은 원본 표지·공고 대조가 필요하다.\\",\\n      \\"conditions\\": \\"파일명만으로 발주기관을 확정하지 않는다. 상주 장소와 발주기관 역할을 구별한다.\\",\\n      \\"source_reference\\": \\"문단 25 외 3곳\\\\nD01-02-01~04\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D01\\",\\n      \\"field_id\\": \\"purpose\\",\\n      \\"label\\": \\"목적\\",\\n      \\"expected\\": \\"세무 전문업체의 교육·신고자료 검토로 업무를 효율화하고, 세법 정보·상담으로 연말정산 오류를 최소화한다.\\",\\n      \\"conditions\\": \\"효율적 업무처리와 신고오류 최소화 목적을 모두 포함한다.\\",\\n      \\"source_reference\\": \\"문단 80 외 1곳\\\\nD01-03-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D01\\",\\n      \\"field_id\\": \\"scope_requirements\\",\\n      \\"label\\": \\"주요 범위·필수요건\\",\\n      \\"expected\\": \\"안내문·IU 매뉴얼, 오류 검증, 상담·자료 접수와 입력 보조, 지급명세서·결과보고서, 증빙 편철·제출, 부당공제 후속 처리와 개인정보 보호·자료 반납/파기를 수행한다.\\",\\n      \\"conditions\\": \\"예정인원 4,200명 이내는 추정치이며 변동 가능. 상세 과업표·인력 상주·주 2회 이상 총괄책임자 방문 조건도 확인한다.\\",\\n      \\"source_reference\\": \\"문단 64 외 15곳\\\\nD01-04-01~16\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D01\\",\\n      \\"field_id\\": \\"budget_vat\\",\\n      \\"label\\": \\"예산·VAT\\",\\n      \\"expected\\": \\"사업예산 57,246,000원, 부가가치세 제외. 단가계약으로 실제 대상인원에 대하여 정산·지급한다.\\",\\n      \\"conditions\\": \\"부가세 제외를 포함으로 바꾸지 않는다. 예정 총액과 실제 정산금액을 구별한다.\\",\\n      \\"source_reference\\": \\"문단 72 외 1곳\\\\nD01-05-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D01\\",\\n      \\"field_id\\": \\"duration\\",\\n      \\"label\\": \\"수행기간\\",\\n      \\"expected\\": \\"계약체결일~2027.3.17. 당해 연도 부당 공제자 통보 시 후속절차 지원은 과업기간 이후에도 포함된다.\\",\\n      \\"conditions\\": \\"종료일만 적고 기간 이후 지원 의무를 누락하지 않는다.\\",\\n      \\"source_reference\\": \\"문단 61 외 2곳\\\\nD01-06-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D01\\",\\n      \\"field_id\\": \\"eligibility\\",\\n      \\"label\\": \\"참가·수행자격\\",\\n      \\"expected\\": \\"[부분확인]\\\\n총괄책임자는 세무사 자격과 연말정산 3년 이상 경력. 상주인력은 연말정산 2년 이상 경력자 3명 이상이다.\\",\\n      \\"conditions\\": \\"이는 수행인력 요건이다. 별도의 입찰 참가업체 자격이 모두 확인됐다고 쓰지 않는다.\\",\\n      \\"source_reference\\": \\"문단 165 외 2곳\\\\nD01-07-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D01\\",\\n      \\"field_id\\": \\"submission_documents\\",\\n      \\"label\\": \\"입찰 제출서류\\",\\n      \\"expected\\": \\"[미확인]\\\\n입찰 제출서류 목록은 현재 추출 본문에서 확인하지 못했다. 착수·준공 제출문서는 별도로 존재한다.\\",\\n      \\"conditions\\": \\"착수계·공정예정표·준공계·결과보고서를 입찰 제출서류로 바꾸지 않는다.\\",\\n      \\"source_reference\\": \\"문단 254 외 7곳\\\\nD01-08-01~08\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D01\\",\\n      \\"field_id\\": \\"submission_deadline\\",\\n      \\"label\\": \\"제출기한\\",\\n      \\"expected\\": \\"[미확인]\\\\n입찰 마감 연월일·시각은 현재 추출 본문에서 확인하지 못했다.\\",\\n      \\"conditions\\": \\"계약 시작 후 7일 이내 착수계 제출, 준공 완료기한을 입찰 마감으로 사용하지 않는다.\\",\\n      \\"source_reference\\": \\"문단 122 외 3곳\\\\nD01-09-01~04\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D01\\",\\n      \\"field_id\\": \\"submission_method\\",\\n      \\"label\\": \\"제출방식\\",\\n      \\"expected\\": \\"[미확인]\\\\n입찰 접수 경로는 현재 추출 본문에서 확인하지 못했다.\\",\\n      \\"conditions\\": \\"착수·준공 문서 제출이나 홈페이지의 서식 게시 위치를 입찰 접수 방법으로 사용하지 않는다.\\",\\n      \\"source_reference\\": \\"문단 256 외 1곳\\\\nD01-10-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D01\\",\\n      \\"field_id\\": \\"evaluation\\",\\n      \\"label\\": \\"평가기준·선정방식\\",\\n      \\"expected\\": \\"[부분확인]\\\\n낙찰자 선정방식은 적격심사(제한적최저가) 낙찰제. 세부 평가항목·배점은 현재 추출 본문에서 확인하지 못했다.\\",\\n      \\"conditions\\": \\"선정방식을 남기되 세부 배점표를 추정하지 않는다.\\",\\n      \\"source_reference\\": \\"문단 74\\\\nD01-11-01\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D02\\",\\n      \\"field_id\\": \\"project_name\\",\\n      \\"label\\": \\"사업명\\",\\n      \\"expected\\": \\"2027년 글로벌 패션브랜드 육성 용역\\",\\n      \\"conditions\\": \\"파일명상의 2026년과 사업명 2027년을 구별한다.\\",\\n      \\"source_reference\\": \\"문단 65\\\\nD02-01-01\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D02\\",\\n      \\"field_id\\": \\"issuing_agency\\",\\n      \\"label\\": \\"발주기관\\",\\n      \\"expected\\": \\"서울특별시. 담당부서는 뷰티패션산업과로 표시되어 있다.\\",\\n      \\"conditions\\": \\"발주기관과 부서·현지 협력기관 CNMI를 혼동하지 않는다.\\",\\n      \\"source_reference\\": \\"문단 157 외 3곳\\\\nD02-02-01~04\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D02\\",\\n      \\"field_id\\": \\"purpose\\",\\n      \\"label\\": \\"목적\\",\\n      \\"expected\\": \\"국내 유망 디자이너 브랜드의 글로벌 인지도를 높이고 현지 마케팅·세일즈로 해외 패션시장 공략을 지원한다.\\",\\n      \\"conditions\\": \\"인지도 향상과 해외 시장 공략 목적을 보존한다.\\",\\n      \\"source_reference\\": \\"문단 73 외 1곳\\\\nD02-03-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D02\\",\\n      \\"field_id\\": \\"scope_requirements\\",\\n      \\"label\\": \\"주요 범위·필수요건\\",\\n      \\"expected\\": \\"2027 아시아 패션 세일즈 프로그램과 밀라노 패션위크 파트너십 프로그램을 운영한다. 전시 콘텐츠·현지 홍보·세일즈·네트워킹 등을 포함한다.\\",\\n      \\"conditions\\": \\"장소는 아시아 쇼룸 등과 밀라노. 세부사항은 제안 및 서울시·CNMI 협의에 따라 변동 가능하다.\\",\\n      \\"source_reference\\": \\"문단 77 외 13곳\\\\nD02-04-01~14\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D02\\",\\n      \\"field_id\\": \\"budget_vat\\",\\n      \\"label\\": \\"예산·VAT\\",\\n      \\"expected\\": \\"용역비 650,000,000원, 부가세 포함.\\",\\n      \\"conditions\\": \\"650백만원과 650,000,000원을 같은 금액으로 대조하되 VAT 조건을 보존한다.\\",\\n      \\"source_reference\\": \\"문단 69\\\\nD02-05-01\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D02\\",\\n      \\"field_id\\": \\"duration\\",\\n      \\"label\\": \\"수행기간\\",\\n      \\"expected\\": \\"[표기차이 확인]\\\\n개요는 2026.12~2027.11 중 프로그램 진행기간 10개월. 상반기는 2026.11.30~2027.4.30(5개월), 하반기는 2027.7.1~11.30(5개월)로 기재되어 있다.\\",\\n      \\"conditions\\": \\"개요 시작월과 상반기 시작일의 표기 차이를 숨기거나 임의 수정하지 않는다. 10개월을 연속 일정으로 단정하지 않는다.\\",\\n      \\"source_reference\\": \\"문단 66 외 2곳\\\\nD02-06-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D02\\",\\n      \\"field_id\\": \\"eligibility\\",\\n      \\"label\\": \\"참가·수행자격\\",\\n      \\"expected\\": \\"[공고문 참조]\\\\n입찰 참가자격은 공고문 참조로 명시되어 있다.\\",\\n      \\"conditions\\": \\"별도 공고문이 없는 상태에서 업종·실적·지역 요건을 추정하지 않는다.\\",\\n      \\"source_reference\\": \\"문단 123\\\\nD02-07-01\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D02\\",\\n      \\"field_id\\": \\"submission_documents\\",\\n      \\"label\\": \\"입찰 제출서류\\",\\n      \\"expected\\": \\"입찰참가신청서·위임/재직·등록/등기/인감·각종 서약·공동수급 서류(해당 시)와 제안서 공문·수행능력평가서·내용제안서/발표자료·USB·가격제안서를 제출한다.\\",\\n      \\"conditions\\": \\"내용제안서와 요약본 각 11부(원본 각1, 평가용 각10), USB1개. 평가용 익명 처리, 가격제안서 별도 밀봉·날인을 보존한다.\\",\\n      \\"source_reference\\": \\"문단 177 외 26곳\\\\nD02-08-01~27\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D02\\",\\n      \\"field_id\\": \\"submission_deadline\\",\\n      \\"label\\": \\"제출기한\\",\\n      \\"expected\\": \\"[공고문 참조]\\\\n입찰등록·가격투찰 일시 및 제안서 제출 일시·장소는 입찰공고문에 따른다.\\",\\n      \\"conditions\\": \\"문서 작성일·행사일·프로그램 일정을 입찰 마감으로 바꾸지 않는다.\\",\\n      \\"source_reference\\": \\"문단 156 외 1곳\\\\nD02-09-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D02\\",\\n      \\"field_id\\": \\"submission_method\\",\\n      \\"label\\": \\"제출방식\\",\\n      \\"expected\\": \\"등록·가격투찰은 나라장터 전자입찰. 제안서와 가격입찰서는 직접 방문 접수하며 우편·이메일은 불가하다.\\",\\n      \\"conditions\\": \\"G2B 금액과 수기가격제안서 금액이 같아야 한다. 전자입찰 미참여 시 무효라는 조건을 남긴다.\\",\\n      \\"source_reference\\": \\"문단 155 외 6곳\\\\nD02-10-01~07\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D02\\",\\n      \\"field_id\\": \\"evaluation\\",\\n      \\"label\\": \\"평가기준·선정방식\\",\\n      \\"expected\\": \\"기술능력 90%+가격 10%. 정량20점·정성70점·가격10점 구조이며 종합점수 70점 이상을 협상적격자로 선정한다.\\",\\n      \\"conditions\\": \\"다른 RFP의 기술점수 85% 기준을 가져오지 않는다. 동점은 기술점수 우선, 그마저 같으면 추첨이다.\\",\\n      \\"source_reference\\": \\"문단 131 외 6곳\\\\nD02-11-01~07\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D03\\",\\n      \\"field_id\\": \\"project_name\\",\\n      \\"label\\": \\"사업명\\",\\n      \\"expected\\": \\"HF 인사이트 시스템 개발\\",\\n      \\"conditions\\": \\"사업명과 일반적인 AI 시스템 명칭을 혼동하지 않는다.\\",\\n      \\"source_reference\\": \\"문단 11 외 1곳\\\\nD03-01-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D03\\",\\n      \\"field_id\\": \\"issuing_agency\\",\\n      \\"label\\": \\"발주기관\\",\\n      \\"expected\\": \\"한국주택금융공사\\",\\n      \\"conditions\\": \\"표지의 발주기관 표기와 기관명을 함께 확인한다.\\",\\n      \\"source_reference\\": \\"문단 12 외 1곳\\\\nD03-02-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D03\\",\\n      \\"field_id\\": \\"purpose\\",\\n      \\"label\\": \\"목적\\",\\n      \\"expected\\": \\"인터넷 데이터를 룰베이스·머신러닝/딥러닝으로 분석해 공사에 대한 국민 요구와 긍·부정 인식을 파악하는 시스템을 구축한다.\\",\\n      \\"conditions\\": \\"경제·금융 이슈 파악과 공사 관련 인식·요구 분석을 포함한다.\\",\\n      \\"source_reference\\": \\"문단 105 외 4곳\\\\nD03-03-01~05\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D03\\",\\n      \\"field_id\\": \\"scope_requirements\\",\\n      \\"label\\": \\"주요 범위·필수요건\\",\\n      \\"expected\\": \\"인터넷 기사·댓글·카페·블로그·커뮤니티·유튜브 등의 수집, 이슈·연관어·감성 분석, 시각화 보고와 내부 업무시스템 구현을 포함한다.\\",\\n      \\"conditions\\": \\"최소10분 단위 수집, 용역 개시 후 데이터 적재(과거자료 제외), 긍·부정 정확도 최소90% 및 산출·유지 방법론, 지속 수급·알고리즘 교체 가능성을 확인한다.\\",\\n      \\"source_reference\\": \\"문단 131 외 15곳\\\\nD03-04-01~16\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D03\\",\\n      \\"field_id\\": \\"budget_vat\\",\\n      \\"label\\": \\"예산·VAT\\",\\n      \\"expected\\": \\"사업예산 688백만원, 부가세 포함. 개발498백만원+HW구입190백만원으로 구성된다.\\",\\n      \\"conditions\\": \\"총액·개발·HW의 구분을 유지한다. 발주기관 사정에 따른 예산·기간·일정 조정 가능성도 확인한다.\\",\\n      \\"source_reference\\": \\"문단 99 외 2곳\\\\nD03-05-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D03\\",\\n      \\"field_id\\": \\"duration\\",\\n      \\"label\\": \\"수행기간\\",\\n      \\"expected\\": \\"계약일로부터 5개월이며 안정화·시스템모니터링 1개월이 그 안에 포함된다.\\",\\n      \\"conditions\\": \\"5+1=6개월로 바꾸지 않는다. 발주기관 내부 사정에 따른 조정 가능성을 남긴다.\\",\\n      \\"source_reference\\": \\"문단 97 외 1곳\\\\nD03-06-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D03\\",\\n      \\"field_id\\": \\"eligibility\\",\\n      \\"label\\": \\"참가·수행자격\\",\\n      \\"expected\\": \\"입찰등록·법령상 자격, 부정당 제재 해당 없음, SW사업자(1468), 직접생산확인증명서(정보시스템개발서비스 8111159901), 유효한 중소기업·소상공인 확인서 등을 모두 갖춰야 한다.\\",\\n      \\"conditions\\": \\"20억원 미만으로 대기업·중견 SW사업자 참여 제한. 공동이행만 허용, 대표 포함5개사 이내·최소지분10%, 기타는 공고문 참조.\\",\\n      \\"source_reference\\": \\"문단 2616 외 12곳\\\\nD03-07-01~13\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D03\\",\\n      \\"field_id\\": \\"submission_documents\\",\\n      \\"label\\": \\"입찰 제출서류\\",\\n      \\"expected\\": \\"[부분확인]\\\\n전체 제출서류 목록은 공고문에 따르며 본문에 정성·정량 제출서류 표가 별도로 있다.\\",\\n      \\"conditions\\": \\"사업자등록·등기·인감·위임/재직·완납증명·SW신고·청렴·하도급·개인정보·공동수급·직접생산 서류, 신용평가·실적·중소기업 확인 등을 해당 조건에 따라 구별한다.\\",\\n      \\"source_reference\\": \\"문단 2774 외 21곳\\\\nD03-08-01~22\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D03\\",\\n      \\"field_id\\": \\"submission_deadline\\",\\n      \\"label\\": \\"제출기한\\",\\n      \\"expected\\": \\"[공고문 참조]\\\\n제출 기한은 입찰공고문에 따른다.\\",\\n      \\"conditions\\": \\"사업기간·제안발표 일정을 입찰 마감으로 바꾸지 않는다.\\",\\n      \\"source_reference\\": \\"문단 2774\\\\nD03-09-01\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D03\\",\\n      \\"field_id\\": \\"submission_method\\",\\n      \\"label\\": \\"제출방식\\",\\n      \\"expected\\": \\"[부분확인]\\\\n제출 장소·전체 제출사항은 공고문에 따른다. 가격입찰은 국가종합전자조달시스템을 이용한다.\\",\\n      \\"conditions\\": \\"계약 체결 후 제안서를 PDF로 공사에 제출한다는 조건을 입찰 접수 경로와 구별한다. 원본/사본 확인·날인 조건을 확인한다.\\",\\n      \\"source_reference\\": \\"문단 2650 외 3곳\\\\nD03-10-01~04\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D03\\",\\n      \\"field_id\\": \\"evaluation\\",\\n      \\"label\\": \\"평가기준·선정방식\\",\\n      \\"expected\\": \\"종합100점=기술90점+가격10점. 가격이 예정가격 이하이며 기술 배점의85% 이상인 업체가 협상적격자다. 기술평가는 순위 간3점 차등점수제를 적용한다.\\",\\n      \\"conditions\\": \\"85%는 전체100점 기준이 아니다. 협상은 합산 고득점 순이며 차순위 협상·재공고 조건을 확인한다.\\",\\n      \\"source_reference\\": \\"문단 2644 외 9곳\\\\nD03-11-01~10\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D04\\",\\n      \\"field_id\\": \\"project_name\\",\\n      \\"label\\": \\"사업명\\",\\n      \\"expected\\": \\"부동산 등기정보 시스템 구축\\",\\n      \\"conditions\\": \\"등기부 발급 서비스와 시스템 구축 사업명을 구별한다.\\",\\n      \\"source_reference\\": \\"문단 6 외 1곳\\\\nD04-01-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D04\\",\\n      \\"field_id\\": \\"issuing_agency\\",\\n      \\"label\\": \\"발주기관\\",\\n      \\"expected\\": \\"한국주택금융공사. 본문 청렴계약 이행서약서의 발주자 문구와 원본·공고를 함께 대조한다.\\",\\n      \\"conditions\\": \\"다른 사업명이 남은 현장실사 서식은 이 사업의 사업명 근거로 사용하지 않는다.\\",\\n      \\"source_reference\\": \\"문단 1900 외 3곳\\\\nD04-02-01~04\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D04\\",\\n      \\"field_id\\": \\"purpose\\",\\n      \\"label\\": \\"목적\\",\\n      \\"expected\\": \\"수기 발급·육안 확인하던 등기사항전부증명서와 토지·건축물대장 정보를 데이터로 받아 업무를 전산화·자동화한다.\\",\\n      \\"conditions\\": \\"부동산뿐 아니라 법인 등기와 대장 정보도 포함한다.\\",\\n      \\"source_reference\\": \\"문단 114 외 2곳\\\\nD04-03-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D04\\",\\n      \\"field_id\\": \\"scope_requirements\\",\\n      \\"label\\": \\"주요 범위·필수요건\\",\\n      \\"expected\\": \\"부동산·법인 등기사항전부증명서와 토지·건축물대장 데이터를 실시간으로 제공한다.\\",\\n      \\"conditions\\": \\"연간 등기발급 예상318,000건은 가격 제안의 참고량이며 대금은 제공량에 따라 달라진다. 세부 기능요건은 원문 요구사항 절도 확인한다.\\",\\n      \\"source_reference\\": \\"문단 118 외 2곳\\\\nD04-04-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D04\\",\\n      \\"field_id\\": \\"budget_vat\\",\\n      \\"label\\": \\"예산·VAT\\",\\n      \\"expected\\": \\"사업예산은 280백만원 이내이며 부가세 포함. 대금은 예산 이내에서 데이터 제공량에 따라 증감한다.\\",\\n      \\"conditions\\": \\"상한인 ‘이내’를 보존한다. 예상318,000건과 확정 납품·정산량을 혼동하지 않는다.\\",\\n      \\"source_reference\\": \\"문단 122 외 2곳\\\\nD04-05-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D04\\",\\n      \\"field_id\\": \\"duration\\",\\n      \\"label\\": \\"수행기간\\",\\n      \\"expected\\": \\"계약체결일로부터365일. 외부 정보연계 협의 지연 시 단계적 오픈 및 과업 조정이 가능하다.\\",\\n      \\"conditions\\": \\"365일을 임의로12개월로 치환하지 않고 조정 조건을 함께 기록한다.\\",\\n      \\"source_reference\\": \\"문단 124 외 1곳\\\\nD04-06-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D04\\",\\n      \\"field_id\\": \\"eligibility\\",\\n      \\"label\\": \\"참가·수행자격\\",\\n      \\"expected\\": \\"입찰등록·관련 자격, 컴퓨터 관련 서비스사업 SW사업자 신고와 직접생산증명서가 필요하며 대기업·중견기업 참여가 제한된다.\\",\\n      \\"conditions\\": \\"공동이행만 허용, 대표 포함5개사 이내·최소지분10%. 기타는 공고문 참조.\\",\\n      \\"source_reference\\": \\"문단 577 외 12곳\\\\nD04-07-01~13\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D04\\",\\n      \\"field_id\\": \\"submission_documents\\",\\n      \\"label\\": \\"입찰 제출서류\\",\\n      \\"expected\\": \\"[부분확인]\\\\n전체 제출서류·일시·장소는 공고문 참고. 본문 정성 제안서 합본 서류와 원본 제출 조건을 확인한다.\\",\\n      \\"conditions\\": \\"사업자등록·등기·인감·위임/재직·완납·SW신고·청렴·일반현황·하도급·개인정보 등 해당 서류를 구별한다.\\",\\n      \\"source_reference\\": \\"문단 628 외 19곳\\\\nD04-08-01~20\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D04\\",\\n      \\"field_id\\": \\"submission_deadline\\",\\n      \\"label\\": \\"제출기한\\",\\n      \\"expected\\": \\"[공고문 참조]\\\\n제출 일시는 공고문 참고이다.\\",\\n      \\"conditions\\": \\"입찰 마감 날짜·시간을 사업기간에서 추정하지 않는다.\\",\\n      \\"source_reference\\": \\"문단 628\\\\nD04-09-01\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D04\\",\\n      \\"field_id\\": \\"submission_method\\",\\n      \\"label\\": \\"제출방식\\",\\n      \\"expected\\": \\"[부분확인]\\\\n접수 일시·장소 등은 공고문에 따른다. 가격입찰은 국가종합전자조달시스템을 이용한다.\\",\\n      \\"conditions\\": \\"원본 제출·사본 확인날인 조건과 계약 후 제안서 CD3장 제출을 입찰 제출 방식과 구별한다.\\",\\n      \\"source_reference\\": \\"문단 604 외 4곳\\\\nD04-10-01~05\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D04\\",\\n      \\"field_id\\": \\"evaluation\\",\\n      \\"label\\": \\"평가기준·선정방식\\",\\n      \\"expected\\": \\"제한경쟁·협상계약. 기술90점+가격10점, 기술 배점의85% 이상을 협상적격자로 선정한다.\\",\\n      \\"conditions\\": \\"합산 동점은 기술점수, 그마저 같으면 큰 배점 세부항목 점수 순이다. 기술85%를 전체85점으로 바꾸지 않는다.\\",\\n      \\"source_reference\\": \\"문단 132 외 7곳\\\\nD04-11-01~08\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D05\\",\\n      \\"field_id\\": \\"project_name\\",\\n      \\"label\\": \\"사업명\\",\\n      \\"expected\\": \\"AI 기반 관광 B2C 플랫폼 통합 ISP 수립\\",\\n      \\"conditions\\": \\"ISP 수립을 실제 플랫폼 구축·운영 계약으로 바꾸지 않는다.\\",\\n      \\"source_reference\\": \\"문단 19 외 2곳\\\\nD05-01-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D05\\",\\n      \\"field_id\\": \\"issuing_agency\\",\\n      \\"label\\": \\"발주기관\\",\\n      \\"expected\\": \\"한국관광공사. 담당부서는 관광AX기획팀으로 표시되어 있다.\\",\\n      \\"conditions\\": \\"설명회 장소와 제출 우편주소·담당부서의 역할을 구별한다.\\",\\n      \\"source_reference\\": \\"문단 29 외 4곳\\\\nD05-02-01~05\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D05\\",\\n      \\"field_id\\": \\"purpose\\",\\n      \\"label\\": \\"목적\\",\\n      \\"expected\\": \\"B2C 채널 기능·VOC 창구를 통합해 AX 기반과 운영 효율을 높이고, AI 여행비서 역할·1330 연계를 설계하며, 데이터 환류·개방 기반의 공공 AI 인프라를 조성한다.\\",\\n      \\"conditions\\": \\"플랫폼 통합·AI 여행비서·데이터 활용의 세 가지 목적을 구별한다.\\",\\n      \\"source_reference\\": \\"문단 52 외 8곳\\\\nD05-03-01~09\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D05\\",\\n      \\"field_id\\": \\"scope_requirements\\",\\n      \\"label\\": \\"주요 범위·필수요건\\",\\n      \\"expected\\": \\"11개 B2C 채널 현황·통합전략, 통합회원·개인정보 정책, 플랫폼/데이터 활용 구조, 단계별 로드맵, 목업과 구축 RFP를 마련한다.\\",\\n      \\"conditions\\": \\"현황 분석·목표모델·이행계획과 후속 구축용RFP를 구분한다. 사업 종료 이후 예산대응 자료보완 의무도 확인한다.\\",\\n      \\"source_reference\\": \\"문단 61 외 8곳\\\\nD05-04-01~09\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D05\\",\\n      \\"field_id\\": \\"budget_vat\\",\\n      \\"label\\": \\"예산·VAT\\",\\n      \\"expected\\": \\"과업예산248,775,000원, VAT 포함. 제안가격에는 사업 관련 모든 세금 등을 포함한다.\\",\\n      \\"conditions\\": \\"본 용역 예산과 향후 플랫폼 구축 예상예산을 섞지 않는다.\\",\\n      \\"source_reference\\": \\"문단 81 외 2곳\\\\nD05-05-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D05\\",\\n      \\"field_id\\": \\"duration\\",\\n      \\"label\\": \\"수행기간\\",\\n      \\"expected\\": \\"계약체결일부터2026.10.30까지이며, 과업 종료 후 정산 및 결과보고서 제출 시까지라는 조건이 붙는다.\\",\\n      \\"conditions\\": \\"달력상 종료일만 적지 않는다. 사업 종료 이후 예산대응 자료보완 조건도 확인한다.\\",\\n      \\"source_reference\\": \\"문단 77 외 1곳\\\\nD05-06-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D05\\",\\n      \\"field_id\\": \\"eligibility\\",\\n      \\"label\\": \\"참가·수행자격\\",\\n      \\"expected\\": \\"G2B SW사업자(컴퓨터 관련 서비스업1468) 등록, 관련 법상 자격과 중소기업 요건 등을 모두 충족해야 한다.\\",\\n      \\"conditions\\": \\"대기업·중견기업·상호출자제한기업집단 참여 제한. 공동이행5개 이하·최소지분10%, 하도급 허용. 유효 확인서·실적 등 제출서류를 원문에서 대조한다.\\",\\n      \\"source_reference\\": \\"문단 1748 외 19곳\\\\nD05-07-01~20\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D05\\",\\n      \\"field_id\\": \\"submission_documents\\",\\n      \\"label\\": \\"입찰 제출서류\\",\\n      \\"expected\\": \\"가격제안서/세부산출1부, 제안서/요약본 각1부, 입찰참가 등록서류 및 부속자료를 제출한다.\\",\\n      \\"conditions\\": \\"우편은USB(PPT/PDF)만 제출·인쇄본 불필요. 입찰참가/신용평가/인력/PM재직/중소기업/공동수급 등 조건별 서류를 빠뜨리지 않는다.\\",\\n      \\"source_reference\\": \\"문단 2078 외 23곳\\\\nD05-08-01~24\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D05\\",\\n      \\"field_id\\": \\"submission_deadline\\",\\n      \\"label\\": \\"제출기한\\",\\n      \\"expected\\": \\"[공고문 참조]\\\\n입찰 제출기한은 입찰공고에 명기된 기한에 따른다.\\",\\n      \\"conditions\\": \\"사전설명회 신청4월29일15시·설명회4월30일14시를 입찰 마감으로 사용하지 않는다. 우편은 마감 전 도착분만 접수한다.\\",\\n      \\"source_reference\\": \\"문단 84 외 5곳\\\\nD05-09-01~06\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D05\\",\\n      \\"field_id\\": \\"submission_method\\",\\n      \\"label\\": \\"제출방식\\",\\n      \\"expected\\": \\"가격은 나라장터 전자입찰. 제안서/부속서류는 전자우편 또는 우편, 사전협의 시 방문 제출 가능하다.\\",\\n      \\"conditions\\": \\"방문 제출 가능 조건·우편 도착 기준·USB 제출 조건을 구별한다. 설명회 신청용 이메일과 제안서 제출 경로를 섞지 않는다.\\",\\n      \\"source_reference\\": \\"문단 2079 외 8곳\\\\nD05-10-01~09\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"D05\\",\\n      \\"field_id\\": \\"evaluation\\",\\n      \\"label\\": \\"평가기준·선정방식\\",\\n      \\"expected\\": \\"기술90점(계량10+비계량80)+가격10점. 기술76.5점(85%) 미만은 협상적격자에서 제외한다.\\",\\n      \\"conditions\\": \\"PM 직접발표, 최고·최저 각1개 제외 평균, 동점은 기술·큰 배점 항목 순. 최종 낙찰업체·실제계약금액은 이 RFP만으로 확정하지 않는다.\\",\\n      \\"source_reference\\": \\"문단 1781 외 9곳\\\\nD05-11-01~10\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R06\\",\\n      \\"field_id\\": \\"project_name\\",\\n      \\"label\\": \\"사업명\\",\\n      \\"expected\\": \\"조합 전산시스템 고도화 사업(전면개편)\\",\\n      \\"conditions\\": \\"표지의 약칭과 사업개요의 정식 사업명을 대조한다.\\",\\n      \\"source_reference\\": \\"PDF p.3\\\\nR06-01-01\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R06\\",\\n      \\"field_id\\": \\"issuing_agency\\",\\n      \\"label\\": \\"발주기관\\",\\n      \\"expected\\": \\"[부분확인]\\\\n건설엔지니어링공제조합. 본문 통합전산시스템 명칭과 조합 조직체계가 근거이며 원본 발주자 표기를 함께 확인한다.\\",\\n      \\"conditions\\": \\"파일명만으로 기관을 확정하지 않는다.\\",\\n      \\"source_reference\\": \\"PDF p.13 외 1곳\\\\nR06-02-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R06\\",\\n      \\"field_id\\": \\"purpose\\",\\n      \\"label\\": \\"목적\\",\\n      \\"expected\\": \\"노후 전산시스템 HW·SW를 개편해 안정적인 운영기반과 업무 효율성·조합원 서비스 만족도를 높인다.\\",\\n      \\"conditions\\": \\"노후 부품·버전 제약 및 대내/대외 기대효과를 구별한다.\\",\\n      \\"source_reference\\": \\"PDF p.3\\\\nR06-03-01\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R06\\",\\n      \\"field_id\\": \\"scope_requirements\\",\\n      \\"label\\": \\"주요 범위·필수요건\\",\\n      \\"expected\\": \\"홈페이지·공제업무관리·경영관리통합 등3개 시스템 전반을 고도화하고 클라우드 서버·DB·보안·기존 운영종료 지원 등 요구사항을 수행한다.\\",\\n      \\"conditions\\": \\"세부 기능·성능·테스트·보안·프로젝트 관리/지원 요구사항은 요구사항 목록과 상세 절에 대조한다.\\",\\n      \\"source_reference\\": \\"PDF p.5 외 2곳\\\\nR06-04-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R06\\",\\n      \\"field_id\\": \\"budget_vat\\",\\n      \\"label\\": \\"예산·VAT\\",\\n      \\"expected\\": \\"[원본 확인 필요]\\\\n추출 원문 예정가격 표기는 “1,189,000,0000원”으로 자릿수/쉼표가 불규칙하다. 직접경비·부가세 등 일체비용 포함이라고 명시되어 있다.\\",\\n      \\"conditions\\": \\"원본 PDF·공고에서 금액을 확인하기 전에는 1,189,000,000원 등으로 임의 정정·환산하지 않는다.\\",\\n      \\"source_reference\\": \\"PDF p.3\\\\nR06-05-01\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R06\\",\\n      \\"field_id\\": \\"duration\\",\\n      \\"label\\": \\"수행기간\\",\\n      \\"expected\\": \\"2025.6~2026.8, 계약체결 후 약15개월. 세부 일정은 전체15개월 내 개발여건에 따라 조정 가능하다.\\",\\n      \\"conditions\\": \\"무상 하자보수는 준공일로부터1년으로 별도 조건이다.\\",\\n      \\"source_reference\\": \\"PDF p.3 외 2곳\\\\nR06-06-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R06\\",\\n      \\"field_id\\": \\"eligibility\\",\\n      \\"label\\": \\"참가·수행자격\\",\\n      \\"expected\\": \\"입찰 제한기간이 아닌 SW사업자로, 최근5년 내 금융·보험·공제 분야 계약금액10억원(VAT포함) 이상 개발실적1건 이상을 갖춰야 한다. 공동수급·하도급은 불허한다.\\",\\n      \\"conditions\\": \\"참여인력은 소속1년 이상 계속근로 직원, PM은 전체기간 개발현장 상주. 법령의 현행 해석을 덧붙이지 않고 문서 조건으로 검토한다.\\",\\n      \\"source_reference\\": \\"PDF p.42 외 2곳\\\\nR06-07-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R06\\",\\n      \\"field_id\\": \\"submission_documents\\",\\n      \\"label\\": \\"입찰 제출서류\\",\\n      \\"expected\\": \\"[부분확인]\\\\n제안서와 사전 제출할 발표자료, 정량평가용 신용등급 확인서·개발실적 증빙이 본문에 제시된다. 입찰 등록서류 전체 목록은 별도 확인이 필요하다.\\",\\n      \\"conditions\\": \\"입찰 시 증빙과 계약체결 시 전 직원 보안유지각서를 구분한다. 제안서 본문200페이지 이내 조건도 확인한다.\\",\\n      \\"source_reference\\": \\"PDF p.42 외 2곳\\\\nR06-08-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R06\\",\\n      \\"field_id\\": \\"submission_deadline\\",\\n      \\"label\\": \\"제출기한\\",\\n      \\"expected\\": \\"[미확인]\\\\n입찰서/제안서 마감 연월일·시각은 현재 추출 원문에서 확인하지 못했다. 발표 일시·장소는 별도 공지다.\\",\\n      \\"conditions\\": \\"2025.5.19 문서일자, 사업 시작월, 발표 별도공지로 입찰 마감을 채우지 않는다.\\",\\n      \\"source_reference\\": \\"PDF p.1 외 1곳\\\\nR06-09-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R06\\",\\n      \\"field_id\\": \\"submission_method\\",\\n      \\"label\\": \\"제출방식\\",\\n      \\"expected\\": \\"[부분확인]\\\\n제안서는 제본 후 제출하고 발표자료는 사전 제출한다. 입찰 접수 경로(방문·우편·전자)는 현재 추출 원문에서 확정하지 못했다.\\",\\n      \\"conditions\\": \\"제본 규격이나 PPT 발표 방식이 입찰 접수 경로를 뜻하지 않는다.\\",\\n      \\"source_reference\\": \\"PDF p.43 외 1곳\\\\nR06-10-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R06\\",\\n      \\"field_id\\": \\"evaluation\\",\\n      \\"label\\": \\"평가기준·선정방식\\",\\n      \\"expected\\": \\"협상계약, 기술90점(정량10+정성80)+가격10점. 기술배점90점의80% 이상이 협상적격 기준이다.\\",\\n      \\"conditions\\": \\"가격평가는 예정가격 대비88% 미만10점,88~95%5점,95%초과2점. 다른 RFP의85% 기준을 가져오지 않는다.\\",\\n      \\"source_reference\\": \\"PDF p.45 외 2곳\\\\nR06-11-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R07\\",\\n      \\"field_id\\": \\"project_name\\",\\n      \\"label\\": \\"사업명\\",\\n      \\"expected\\": \\"2026년 AI연구용 컴퓨팅 지원 프로젝트 GPU 자원 공급 및 유지보수\\",\\n      \\"conditions\\": \\"GPU 구매사업으로 단정하지 않고 자원 공급·유지보수라는 사업명을 보존한다.\\",\\n      \\"source_reference\\": \\"문단 10 외 2곳\\\\nR07-01-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R07\\",\\n      \\"field_id\\": \\"issuing_agency\\",\\n      \\"label\\": \\"발주기관\\",\\n      \\"expected\\": \\"(사)한국정보통신진흥협회가 운영기관으로 명시되어 있다.\\",\\n      \\"conditions\\": \\"원문 역할명은 운영기관이며 이를 명시해 기관과 역할을 함께 기록한다.\\",\\n      \\"source_reference\\": \\"문단 12 외 1곳\\\\nR07-02-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R07\\",\\n      \\"field_id\\": \\"purpose\\",\\n      \\"label\\": \\"목적\\",\\n      \\"expected\\": \\"초거대 AI 연구에 필요한 GPU와 안정적인 연구환경을 제공해 국내 연구진 지원·AI R&D 및 산업 경쟁력을 높인다.\\",\\n      \\"conditions\\": \\"인프라 공급과 연구 촉진 목적을 구별한다.\\",\\n      \\"source_reference\\": \\"문단 61 외 3곳\\\\nR07-03-01~04\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R07\\",\\n      \\"field_id\\": \\"scope_requirements\\",\\n      \\"label\\": \\"주요 범위·필수요건\\",\\n      \\"expected\\": \\"클라우드 GPU 최소960장 이상을 동적할당으로 제공하고 운영·장애대응·연구환경·사용량 모니터링·연구자 지원을 수행한다.\\",\\n      \\"conditions\\": \\"국내 보유자원, A100 포함 시 H100 이상70%(672장)이상. 미할당/준비시간·최소960장 미충족 참여불가·자원 미소진 시 추가제공/크레딧 조건을 확인한다.\\",\\n      \\"source_reference\\": \\"문단 71 외 18곳\\\\nR07-04-01~19\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R07\\",\\n      \\"field_id\\": \\"budget_vat\\",\\n      \\"label\\": \\"예산·VAT\\",\\n      \\"expected\\": \\"사업비15,300,000,000원(153억원), 부가가치세 포함. 모델별 협상 시간당 단가×사용시간으로 월별 또는 지정기간 정산한다.\\",\\n      \\"conditions\\": \\"사용시간은1시간 단위. 누적 사용료가 예산한도에 도달하면 자원 제공기간이 축소될 수 있다.\\",\\n      \\"source_reference\\": \\"문단 57 외 3곳\\\\nR07-05-01~04\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R07\\",\\n      \\"field_id\\": \\"duration\\",\\n      \\"label\\": \\"수행기간\\",\\n      \\"expected\\": \\"계약체결일~2026.12.31. 자원 제공은 최대8개월 또는 예산 소진 시까지이며 추진현황·예산 소진에 따라 종료일 연장 가능하다.\\",\\n      \\"conditions\\": \\"준비시간은 자원 제공시간에서 제외. 미소진 자원은 계약기간 후 추가제공 또는 크레딧 형태로 제공하는 조건을 보존한다.\\",\\n      \\"source_reference\\": \\"문단 55 외 8곳\\\\nR07-06-01~09\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R07\\",\\n      \\"field_id\\": \\"eligibility\\",\\n      \\"label\\": \\"참가·수행자격\\",\\n      \\"expected\\": \\"GPU960장 이상 보유·할당과 동적할당 운영 역량을 가진 기업/컨소시엄. 공급 클라우드 사업자는 CSAP를 획득해야 한다.\\",\\n      \\"conditions\\": \\"컨소시엄은 주관·참여 공급사 모두 CSAP 등 보안인증 필요. 경쟁입찰 자격과 부정당업자 제외 조건을 함께 확인한다.\\",\\n      \\"source_reference\\": \\"문단 347 외 4곳\\\\nR07-07-01~05\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R07\\",\\n      \\"field_id\\": \\"submission_documents\\",\\n      \\"label\\": \\"입찰 제출서류\\",\\n      \\"expected\\": \\"[별도 문서 참조]\\\\n입찰서류 및 제안서 제출은 입찰유의서 참조로 명시되어 있다.\\",\\n      \\"conditions\\": \\"본문 제안서 작성지침·객관적 증빙 요구만으로 전체 입찰 서류 목록을 만들어내지 않는다.\\",\\n      \\"source_reference\\": \\"문단 511 외 1곳\\\\nR07-08-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R07\\",\\n      \\"field_id\\": \\"submission_deadline\\",\\n      \\"label\\": \\"제출기한\\",\\n      \\"expected\\": \\"[별도 문서 참조]\\\\n제출 기한은 입찰유의서 등을 추가 확인해야 하며, 이 RFP만으로 구체 마감일·시각을 확정하지 않는다.\\",\\n      \\"conditions\\": \\"2026.12.31은 사업 종료일이다. 발표회 세부 일정 별도통보와 입찰 마감을 구별한다.\\",\\n      \\"source_reference\\": \\"문단 55 외 2곳\\\\nR07-09-01~03\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R07\\",\\n      \\"field_id\\": \\"submission_method\\",\\n      \\"label\\": \\"제출방식\\",\\n      \\"expected\\": \\"[별도 문서 참조]\\\\n입찰서류 및 제안서 제출 방법은 입찰유의서 참조 범위로 남긴다.\\",\\n      \\"conditions\\": \\"사업수행 산출물의 이메일/USB 제출 조건을 입찰 제안서 제출 경로로 바꾸지 않는다.\\",\\n      \\"source_reference\\": \\"문단 327 외 1곳\\\\nR07-10-01~02\\"\\n    },\\n    {\\n      \\"sample_id\\": \\"R07\\",\\n      \\"field_id\\": \\"evaluation\\",\\n      \\"label\\": \\"평가기준·선정방식\\",\\n      \\"expected\\": \\"일반경쟁·협상계약이며 기술평가100%. 기술배점85% 이상이 협상적격 기준이다.\\",\\n      \\"conditions\\": \\"사업수행능력5·전략방법론25·사업관리등50·연구자지원20. 최고·최저각1개 제외 평균, 동점처리와 사업관리 우선 조건도 확인한다.\\",\\n      \\"source_reference\\": \\"문단 360 외 26곳\\\\nR07-11-01~27\\"\\n    }\\n  ],\\n  \\"reference_status\\": \\"draft_for_source_comparison\\",\\n  \\"human_review_is_not_prefilled\\": true\\n}\\n", "phase5_validation/cache_revalidation.py": "\\"\\"\\"Read-only replay of failed summary batches. No generate, SDK, or API calls.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\n\\n\\ndef revalidate_cached_batches(env, result_path, component, plan):\\n    from rfp_phase4.citation_recovery import validate_with_quote_recovery\\n    from rfp_phase5.fields import FIELD_QUESTION, FIELD_SCHEMA, parse_field_claim\\n    from rfp_phase5.summary import TASK\\n\\n    path = Path(result_path).expanduser().resolve()\\n    saved = json.loads(path.read_text(encoding=\'utf-8\'))\\n    if saved.get(\'doc_id\') != plan[\'doc_id\'] or saved.get(\'plan_fingerprint\') != plan[\'plan_fingerprint\']:\\n        raise ValueError(\'기존 결과의 문서·질문·요약 계획이 현재 요청과 다릅니다.\')\\n    if saved.get(\'source_chunk_ids\') != plan[\'source_chunk_ids\']:\\n        raise ValueError(\'기존 결과와 현재 원문의 전체 청크 범위가 다릅니다.\')\\n    by_id = {b[\'batch_id\']: b for b in plan[\'batches\']}\\n    outcomes = []\\n    llm = component[\'llm\']\\n    before = llm.usage_report()\\n    for old in saved.get(\'batch_outcomes\', []):\\n        if old.get(\'reason\') != \'source_citation_validation_failed\':\\n            continue\\n        batch = by_id.get(old.get(\'batch_id\'))\\n        if batch is None or old.get(\'chunk_ids\') != [c[\'chunk_id\'] for c in batch[\'chunks\']]:\\n            raise ValueError(\'기존 실패 배치의 청크·순서가 현재 계획과 다릅니다.\')\\n        prepared = llm._prepare(TASK, FIELD_QUESTION, batch[\'chunks\'], schema=FIELD_SCHEMA)\\n        if old.get(\'request_fingerprint\') != prepared[\'fingerprint\']:\\n            raise ValueError(\'모델·설정·질문·원문이 기존 API 요청과 다릅니다. 재사용하지 않았습니다.\')\\n        response = llm._cache_read(prepared)  # Read-only checksum-bound lookup; no _guard or SDK.\\n        item = {\'batch_id\': batch[\'batch_id\'], \'chunk_count\': len(batch[\'chunks\']),\\n            \'original_request_fingerprint\': prepared[\'fingerprint\'],\\n            \'original_api_usage\': old.get(\'usage\', {}), \'additional_api_calls\': 0}\\n        if response is None:\\n            item.update(status=\'cache_not_available\', valid=False,\\n                        explanation=\'원래 응답 캐시를 찾지 못했습니다. 진단용 인용만으로 답변을 재구성하지 않았습니다.\')\\n        else:\\n            if (response.get(\'request_fingerprint\') != prepared[\'fingerprint\']\\n                or response.get(\'execution_mode\') != \'live_openai\' or response.get(\'live_api\') is not True):\\n                raise ValueError(\'기존 실제 모델 응답의 실행 구분 또는 요청 지문을 확인하지 못했습니다.\')\\n            payload = response[\'payload\']\\n            for claim in payload.get(\'claims\', []):\\n                parse_field_claim(claim)\\n            checked = validate_with_quote_recovery(env[\'registry\'], payload, batch[\'chunks\'],\\n                                                   allowed_doc_ids=[plan[\'doc_id\']])\\n            item.update(status=\'cached_batch_revalidated\' if checked[\'valid\'] else \'validation_failed\',\\n                valid=checked[\'valid\'], claims_revalidated=len(checked[\'claims\']),\\n                citation_recovery=checked[\'citation_recovery\'],\\n                citation_diagnostics=checked[\'citation_diagnostics\'],\\n                validation_errors=checked.get(\'errors\', []),\\n                revalidated_claims=checked[\'claims\'] if checked[\'valid\'] else [])\\n        outcomes.append(item)\\n    after = llm.usage_report()\\n    if before != after:\\n        raise RuntimeError(\'읽기 전용 재검증에서 사용량 카운터가 변경됐습니다.\')\\n    return {\'version\': \'phase5-cache-source-revalidation-v1\', \'original_result_path\': str(path),\\n        \'doc_id\': plan[\'doc_id\'], \'document_chunk_count\': plan[\'chunk_count\'],\\n        \'batches\': outcomes, \'revalidated_batch_count\': sum(b[\'valid\'] for b in outcomes),\\n        \'full_document_complete\': False, \'answer_semantic_quality_verified\': False,\\n        \'additional_api_calls\': 0, \'cache_or_original_result_modified\': False,\\n        \'scope\': \'원래 API 응답이 캐시에 있을 때 실패한 배치의 인용만 재검증합니다. 미실행 배치의 결과나 정답 판정을 만들지 않습니다.\'}\\n", "phase5_validation/test_cache_revalidation.py": "\\"\\"\\"Synthetic cache records exercise source replay only, never model accuracy.\\"\\"\\"\\nfrom dataclasses import asdict\\nimport copy\\nimport importlib.util\\nimport json\\nfrom pathlib import Path\\nimport tempfile\\nimport unittest\\nfrom unittest.mock import patch\\n\\nimport test_phase5_summary as fixture\\nfrom rfp_phase4.llm import OpenAIGroundedLLM, LLMConfig\\nfrom rfp_phase4.evidence import EvidenceRegistry\\nfrom rfp_phase5.fields import FIELD_SCHEMA, FIELD_QUESTION\\nfrom rfp_phase5.summary import FieldSummarizer, TASK\\n\\nspec = importlib.util.spec_from_file_location(\'cache_revalidation_under_test\', Path(__file__).with_name(\'cache_revalidation.py\'))\\nm = importlib.util.module_from_spec(spec)\\nspec.loader.exec_module(m)\\n\\n\\nclass CacheSourceReplayTests(unittest.TestCase):\\n    def setUp(self):\\n        self.tmp = tempfile.TemporaryDirectory(); self.addCleanup(self.tmp.cleanup)\\n        self.root = Path(self.tmp.name)\\n        docs, chunks = fixture.corpus(1)\\n        self.env = {\'root\': self.root, \'index\': fixture.Index(chunks), \'registry\': EvidenceRegistry(docs, chunks)}\\n        self.llm = OpenAIGroundedLLM(self.root/\'cache\', config=LLMConfig(enabled=True, max_output_tokens=6000),\\n                                   allowed_sources={\'d1\': \'a\'*64})\\n        self.component = {\'llm\': self.llm}\\n        self.plan = FieldSummarizer(self.env[\'index\'], self.llm, self.env[\'registry\']).plan(\'d1\')\\n        self.prepared = self.llm._prepare(TASK, FIELD_QUESTION, self.plan[\'batches\'][0][\'chunks\'], schema=FIELD_SCHEMA)\\n        # Entirely synthetic response, deliberately marked as such by this test\'s name/docstring.\\n        payload = {\'status\': \'answered\', \'claims\': [{\'doc_id\': \'d1\', \'category\': \'budget\',\\n            \'text\': \'[budget_vat|fact] 원문 인용 연결을 검사하는 합성 테스트 주장\',\\n            \'citations\': [{\'chunk_id\': chunks[0][\'chunk_id\'], \'quote\': chunks[0][\'text\'][:95]}]}],\\n            \'missing_information\': []}\\n        self.response = {\'payload\': payload, \'request_fingerprint\': self.prepared[\'fingerprint\'],\\n            \'execution_mode\': \'live_openai\', \'live_api\': True, \'cache_hit\': False, \'usage\': {}}\\n        self.saved = {\'doc_id\': \'d1\', \'plan_fingerprint\': self.plan[\'plan_fingerprint\'],\\n            \'source_chunk_ids\': self.plan[\'source_chunk_ids\'], \'batch_outcomes\': [{\\n                \'batch_id\': \'map-0001\', \'chunk_ids\': self.plan[\'source_chunk_ids\'],\\n                \'reason\': \'source_citation_validation_failed\', \'request_fingerprint\': self.prepared[\'fingerprint\']}]}\\n        self.path = self.root/\'result.json\'\\n        self.write_saved()\\n\\n    def write_saved(self):\\n        self.path.write_text(json.dumps(self.saved), encoding=\'utf-8\')\\n\\n    def run_replay(self):\\n        with patch(\'socket.socket\', side_effect=AssertionError(\'network prohibited\')), patch.object(self.llm, \'generate\', side_effect=AssertionError(\'generate prohibited\')):\\n            return m.revalidate_cached_batches(self.env, self.path, self.component, self.plan)\\n\\n    def test_matching_cache_replays_without_key_sdk_or_calls_and_preserves_bytes(self):\\n        self.llm._cache_write(self.prepared, self.response)\\n        before = self.path.read_bytes(), self.llm._cache_path().read_bytes()\\n        with patch.dict(\'os.environ\', {}, clear=True): r = self.run_replay()\\n        self.assertTrue(r[\'batches\'][0][\'valid\'])\\n        self.assertEqual(r[\'additional_api_calls\'], 0)\\n        self.assertEqual(r[\'batches\'][0][\'claims_revalidated\'], 1)\\n        self.assertFalse(r[\'full_document_complete\'])\\n        self.assertFalse(r[\'answer_semantic_quality_verified\'])\\n        self.assertEqual(before, (self.path.read_bytes(), self.llm._cache_path().read_bytes()))\\n\\n    def test_missing_cache_does_not_reconstruct_claims_or_generate(self):\\n        r = self.run_replay()\\n        self.assertEqual(r[\'batches\'][0][\'status\'], \'cache_not_available\')\\n        self.assertEqual(r[\'revalidated_batch_count\'], 0)\\n        self.assertNotIn(\'revalidated_claims\', r[\'batches\'][0])\\n\\n    def test_request_fingerprint_mismatch_is_rejected(self):\\n        self.saved[\'batch_outcomes\'][0][\'request_fingerprint\'] = \'b\'*64; self.write_saved()\\n        with self.assertRaises(ValueError): self.run_replay()\\n\\n    def test_wrong_document_or_source_ids_are_rejected(self):\\n        for key, value in [(\'doc_id\', \'other\'), (\'source_chunk_ids\', [\'other\'])]:\\n            with self.subTest(key=key):\\n                original = copy.deepcopy(self.saved)\\n                self.saved[key] = value; self.write_saved()\\n                with self.assertRaises(ValueError): self.run_replay()\\n                self.saved = original; self.write_saved()\\n\\n    def test_fabricated_quote_is_still_rejected(self):\\n        self.response[\'payload\'][\'claims\'][0][\'citations\'][0][\'quote\'] = \'조작된 사업비 999조원\'\\n        self.llm._cache_write(self.prepared, self.response)\\n        r = self.run_replay()\\n        self.assertFalse(r[\'batches\'][0][\'valid\'])\\n        self.assertEqual(r[\'batches\'][0][\'revalidated_claims\'], [])\\n\\n    def test_test_client_namespace_cannot_be_accepted_as_actual_model(self):\\n        self.response[\'execution_mode\'] = \'injected_test_client\'\\n        self.llm._cache_write(self.prepared, self.response)\\n        with self.assertRaises(Exception): self.run_replay()\\n\\n\\nif __name__ == \'__main__\': unittest.main()\\n"}')
OLD_SHA256 = json.loads('{"rfp_phase4/citation_recovery.py": "70ecbf0b02f3c17887538cdea79891d560a19dd16ca55cc79c8c9bd0deca406e", "rfp_phase4/summarization.py": "2e88ee833422d16b284af3707f771327d3c36acfca556bb8a88feda051e4c7cd", "tests/test_phase4_citation_split.py": "a2d931fba7d6c9122feaa5d1703372ebbbc0a66e35267135be6d6bbb450831c2", "phase5_validation/final_validation.py": "d90c5465a07d79ecc6c81a7bf9ddf8b14cc44012ca9751b6292b76d622ec8e32", "phase5_validation/test_final_validation.py": "8e4a17e306bb35d66e16f3aade6a1eb0b56394c19e6a0c7440a58ed0a4ff40a5", "phase5_validation/references.json": "1f2ff16a66c36d700882cd61ea3878ac7c77952b2eb7dde18d315778fad03f90"}')
from datetime import datetime, timezone
import shutil

PATCH_ID = 'phase5-literal-reference-v4'

def replace_once(text, old, new):
    if text.count(old) != 1:
        raise RuntimeError(
            '수정 대상 코드의 버전이 다릅니다. 파일을 변경하지 않았습니다.'
        )
    return text.replace(old, new, 1)


relative = 'rfp_phase4/citation_recovery.py'
old = PATCH_FILES[relative]

new = replace_once(
    old,
    "RECOVERY_POLICY = 'unique_whitespace_or_bounded_whole_line_fragments_v3'",
    "RECOVERY_POLICY = 'bounded_lines_and_literal_reference_rebuild_v4'"
)

helper = '''
def _literal_reference_candidates(context, claim, citation):
    cid, quote = citation.get('chunk_id'), citation.get('quote')

    if set(citation) != {'chunk_id', 'quote'} or not isinstance(cid, str):
        return []˙

    # 정상 형식의 ID를 임의로 다른 청크에 연결하지 않습니다.
    if re.fullmatch(r'rfp:[0-9a-f]{16}:[0-9a-f]{24}', cid):
        return []

    chunks = [
        c for c in context.values()
        if c['doc_id'] == claim.get('doc_id')
    ]
    hashes = {c['sha256'] for c in chunks}

    if len(hashes) != 1:
        return []
    if not cid.startswith('rfp:' + next(iter(hashes))[:12]):
        return []
    if not isinstance(quote, str) or not quote.strip():
        return []

    # 현재 배치·같은 문서에서 원문이 정확히 일치하는 경우만 사용합니다.
    matches = [
        c['chunk_id'] for c in chunks
        if quote in c['text']
    ]

    # 반복되는 구절은 하나를 추측하지 않고 모든 일치 청크를 표시합니다.
    return matches if 1 <= len(matches) <= 8 else []

'''

new = replace_once(
    new,
    '\ndef validate_with_quote_recovery(',
    '\n' + helper + '\ndef validate_with_quote_recovery('
)

new = replace_once(
    new,
    "            if chunk is None:\n"
    "                item['outcome'] = 'chunk_not_in_supplied_context'",
    """            if chunk is None:
                matches = _literal_reference_candidates(context, claim, citation)
                item['outcome'] = 'chunk_not_in_supplied_context'
                if matches:
                    replacement_citations.pop()
                    replacement_citations.extend(
                        {'chunk_id': c, 'quote': quote}
                        for c in matches
                    )
                    item.update(
                        outcome='reference_rebuilt_from_literal_matches',
                        original_chunk_id=cid,
                        candidate_chunk_ids=matches,
                        reference_rebuilt=True,
                        requires_reference_review=True,
                        intended_original_location_inferred=False,
                        reference_quote_unchanged=True
                    )
                    corrections += 1"""
)

new = replace_once(
    new,
    "'corrected_citations': corrections, 'split_citations':splits}",
    "'corrected_citations': corrections, 'split_citations':splits, "
    "'reference_rebindings': sum("
    "bool(d.get('reference_rebuilt')) for d in diagnostics)}"
)

OLD_SHA256[relative] = hashlib.sha256(old.encode('utf-8')).hexdigest()
PATCH_FILES[relative] = new


relative = 'phase5_validation/final_validation.py'
old = PATCH_FILES[relative]

new = replace_once(
    old,
    "VERSION = 'phase5-final-validation-v2'",
    "VERSION = 'phase5-final-validation-v3'"
)

extra = '''
        for batch in result.get('batch_outcomes', []):
            for item in batch.get('citation_diagnostics', []):
                if (
                    item.get('claim_field_id') == ref['field_id']
                    and item.get('requires_reference_review')
                ):
                    omitted.append(
                        '원래 청크 ID 오류; 현재 배치에서 인용문이 '
                        '정확히 일치하는 청크를 모두 표시: '
                        + str(item.get('original_chunk_id'))
                        + ' -> '
                        + ', '.join(item.get('candidate_chunk_ids', []))
                        + ' | '
                        + str(item.get('model_quote', ''))
                    )
'''

anchor = "        hints = '\\n'.join([hints, *omitted]).strip()"
new = replace_once(new, anchor, extra + anchor)

OLD_SHA256[relative] = hashlib.sha256(old.encode('utf-8')).hexdigest()
PATCH_FILES[relative] = new

changes = []
for relative, content in PATCH_FILES.items():
    target = (ROOT/relative).resolve()
    if not target.is_relative_to(ROOT):
        raise ValueError('수정 경로가 프로젝트를 벗어납니다.')
    expected = hashlib.sha256(content.encode('utf-8')).hexdigest()
    if target.exists():
        actual = hashlib.sha256(target.read_bytes()).hexdigest()
        if actual not in {expected, OLD_SHA256.get(relative)}:
            raise RuntimeError('직접 수정한 파일을 보존하고 중단합니다: '+relative)
        if actual == expected:
            continue
    changes.append((relative, content))
backup = None
if changes:
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    backup = ROOT/'phase5_outputs/patch_backups'/('citation_v3_'+stamp)
    for relative, content in changes:
        target = ROOT/relative
        if target.exists():
            old = backup/relative
            old.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(target, old)
    for relative, content in changes:
        target = ROOT/relative
        target.parent.mkdir(parents=True, exist_ok=True)
        temporary = target.with_suffix(target.suffix+'.tmp')
        temporary.write_text(content, encoding='utf-8')
        temporary.replace(target)
# A previous Run All may have loaded these aliases. No model clients are invoked.
if changes:
    import importlib
    for name in ['rfp_phase4.citation_recovery','rfp_phase4.summarization','rfp_phase4.rag',
                 'rfp_phase5.summary','rfp_phase5.runtime']:
        if name in sys.modules:
            importlib.reload(sys.modules[name])
VALIDATION_DIR = ROOT/'phase5_validation'
print(json.dumps({'patch_id': PATCH_ID, 'modified_files': [r for r,c in changes],
    'backup_directory': str(backup) if backup else None, 'api_calls': 0,
    'data_index_cache_or_original_results_modified': False}, ensure_ascii=False, indent=2))


{
  "patch_id": "phase5-literal-reference-v4",
  "modified_files": [],
  "backup_directory": null,
  "api_calls": 0,
  "data_index_cache_or_original_results_modified": false
}


## 2. 환경·문서·전체 입력 계획

7건은 개발 5건과 최종 검증 2건으로 고정합니다. 최종 2건의 기준 내용은 개발 수정에 사용하지 않습니다.
기준표 내용은 모델 요청에 넣지 않습니다. 기존 holdout 질문·정답 파일도 읽지 않습니다.

HF 인사이트는 643,323 byte가 필요해 기존 문서별 600,000 byte 상한을 넘습니다. 이번 검증에서는
해당 문서에만 650,000 byte를 메모리상 적용합니다. 다른 문서는 기존 상한을 유지하고 설정 파일은 보존합니다.
모든 문서의 실제 계획이 먼저 표시되며, 900,000 byte를 넘는 문서는 잘라 보내지 않고 중단합니다.


In [3]:
from rfp_phase5.runtime import bootstrap_phase5
from rfp_phase4.runtime import environment_report

environment = environment_report(ROOT)
print('Python 버전:', environment['python_version'])
for name, expected in environment['expected_dependencies'].items():
    print(name, '저장=', expected, '/ 현재=', environment['current_dependencies'].get(name))
if not environment['compatible']:
    raise RuntimeError('저장 인덱스와 커널 버전이 다릅니다. 이전에 통과한 Phase 3 .venv/bin/python 커널을 선택하세요. 인덱스를 재구축하지 않습니다.')
if 'app' in globals() and isinstance(app, dict) and app.get('index') is not None:
    app['index'].close()
app = bootstrap_phase5(ROOT)
spec = importlib.util.spec_from_file_location('phase5_final_runner', VALIDATION_DIR/'final_validation.py')
validation = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validation)
references = validation.read_json(VALIDATION_DIR/'references.json')
suite = validation.ValidationSuite(app, references, suite_id=SUITE_ID,
    per_doc_ceiling=PER_DOCUMENT_INPUT_CEILING, suite_calls=MAX_SUITE_CALLS,
    suite_bytes=MAX_SUITE_INPUT_BYTES)

def show_json(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))

try:
    from IPython.display import display, Markdown
except ImportError:
    def display(value): print(value)
    def Markdown(value): return value

plan_rows = suite.plan_summary()
display(Markdown('| 표본 | 구분 | 청크 | 캐시 없을 때 최대 요청 | 입력 byte | 문서별 입력 상한 |\n'
    '| --- | --- | ---: | ---: | ---: | ---: |\n' + '\n'.join(
    f"| {r['표본']} | {r['구분']} | {r['청크']} | {r['캐시없을때_최대요청']} | {r['입력byte']:,} | {r['문서별_입력상한']:,} |" for r in plan_rows)))
for r in plan_rows:
    print(r['표본'], r['문서명'])
print('전체 청크:', sum(r['청크'] for r in plan_rows))
print('모두 새로 생성할 경우 최대 요청:', sum(r['캐시없을때_최대요청'] for r in plan_rows))
print('총 계획 입력 byte:', sum(r['입력byte'] for r in plan_rows))
print('byte는 토큰 수·달러 비용이 아닙니다. 실제 요청과 캐시를 다시 검사합니다.')
print('검증 결과 폴더:', suite.folder)
print('코드·프롬프트·원문·설정 고정 fingerprint:', suite.fingerprint)


Python 버전: 3.11.7
numpy 저장= 2.3.5 / 현재= 2.3.5
scipy 저장= 1.17.0 / 현재= 1.17.0
scikit-learn 저장= 1.8.0 / 현재= 1.8.0


| 표본 | 구분 | 청크 | 캐시 없을 때 최대 요청 | 입력 byte | 문서별 입력 상한 |
| --- | --- | ---: | ---: | ---: | ---: |
| D01 | dev | 18 | 1 | 31,301 | 600,000 |
| D02 | dev | 154 | 6 | 188,413 | 600,000 |
| D03 | dev | 442 | 19 | 643,323 | 650,000 |
| D04 | dev | 202 | 11 | 357,781 | 600,000 |
| D05 | dev | 245 | 13 | 418,607 | 600,000 |
| R06 | final | 55 | 4 | 121,311 | 600,000 |
| R07 | final | 51 | 3 | 87,881 | 600,000 |

D01 인천공항시설관리(주)_2026_2026년도 귀속 연말정산 위탁운영 용역_과업내용서.hwp
D02 서울특별시_2026_2027년 글로벌 패션브랜드 육성 용역_제안요청서.hwpx
D03 한국주택금융공사_2023_HF 인사이트 시스템 개발_제안요청서.hwp
D04 한국주택금융공사_2021_부동산 등기정보 시스템 구축_제안요청서.hwp
D05 한국관광공사_2026_AI 기반 관광 B2C 플랫폼 통합 ISP 수립_제안요청서.hwp
R06 건설엔지니어링공제조합_2025_조합 전산시스템 고도화 사업(전면개편)_제안요청서.pdf
R07 한국정보통신진흥협회_2026_2026년 AI연구용 컴퓨팅 지원 프로젝트 GPU 자원 공급 및 유지보수_제안요청서.hwp
전체 청크: 1167
모두 새로 생성할 경우 최대 요청: 57
총 계획 입력 byte: 1848617
byte는 토큰 수·달러 비용이 아닙니다. 실제 요청과 캐시를 다시 검사합니다.
검증 결과 폴더: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase5_local_20261111/phase5_outputs/final_validation/phase5_final_v3
코드·프롬프트·원문·설정 고정 fingerprint: 84de77817fc22ecbc7e24b4ae24e3f045cdce76d81651a711d5ccf7a59b52eac


## 3. 오프라인 실패 처리 검증 — API 호출 없음

없는 인용·다른 문서·필드 변조·일부 청크 누락·요청 상한·API 응답 중단·미검토 판정·최종 단계 진입 차단을 검사합니다.
이 테스트는 실패 경로와 연결을 확인합니다. 모의 응답의 내용을 실제 모델 정확도로 집계하지 않습니다.
같은 고정 실행에서 이미 통과한 테스트 기록은 재사용합니다.


In [4]:
test_path = suite.folder / 'offline_tests.json'
if test_path.exists():
    offline_report = validation.read_json(test_path)
    if offline_report.get('suite_fingerprint') != suite.fingerprint:
        raise RuntimeError('오프라인 테스트의 실행 버전이 다릅니다.')
else:
    offline_report = suite.run_offline_tests()
show_json(offline_report)
if offline_report.get('passed') is not True:
    raise RuntimeError('오프라인 실패 처리 테스트에 실패했습니다. offline_tests.log를 확인하세요. 실제 API 실행은 중단합니다.')


{
  "passed": true,
  "tests_run": 102,
  "failures": 0,
  "errors": 0,
  "skipped": 0,
  "api_calls": 0,
  "suite_fingerprint": "84de77817fc22ecbc7e24b4ae24e3f045cdce76d81651a711d5ccf7a59b52eac"
}


## 3-1. 기존 D02 실패 배치 재검증 — API·SDK·키 없이 캐시 읽기

기존 v1의 D02 첫 배치와 현재 요청 지문을 대조합니다. 실제 응답 캐시가 있으면 원본 응답을 읽어 새 인용 규칙으로 검사합니다.
처음 실행에서 `cached_batch_revalidated`와 `valid=true`가 나오면 첫 배치의 인용 복원이 확인된 것입니다.
이는 154개 청크 중 첫 27개 입력 배치에 대한 검사이며, D02 전체 요약 완료나 의미적 정답 판정은 아닙니다.
캐시가 없으면 `cache_not_available`로 남기고 새 API 요청이나 원래 주장 재구성을 하지 않습니다.


In [5]:
old_result_path = (
    ROOT
    / 'phase5_outputs/final_validation/phase5_final_v2/D02/result.json'
)
if old_result_path.is_file():
    spec_replay = importlib.util.spec_from_file_location('phase5_cache_replay', VALIDATION_DIR/'cache_revalidation.py')
    replay_module = importlib.util.module_from_spec(spec_replay)
    spec_replay.loader.exec_module(replay_module)
    sample = next(s for s in suite.samples if s['sample_id']=='D02')
    replay_component = suite._component(sample, enabled=True)  # Constructor only; no SDK, key, or generate.
    try:
        cache_replay_report = replay_module.revalidate_cached_batches(app, old_result_path,
            replay_component, suite.plans['D02'])
    except Exception as error:
        from rfp_phase4.llm import safe_llm_diagnostic
        cache_replay_report = {'status':'cache_revalidation_blocked', 'additional_api_calls':0,
                              'error_diagnostics':[safe_llm_diagnostic(error)]}
    diagnostic_path = suite.folder/'cache_revalidation/D02.json'
    validation.write_json(diagnostic_path, cache_replay_report)
    compact = [{k:b.get(k) for k in ['batch_id','status','valid','chunk_count','claims_revalidated','additional_api_calls']}
               | {'restored_quotes':(b.get('citation_recovery') or {}).get('corrected_citations',0),
                  'omitted_condition_gaps':sum(len(d.get('omitted_source_gaps',[])) for d in b.get('citation_diagnostics',[]))}
               for b in cache_replay_report.get('batches',[])]
    show_json({'기존_D02_캐시_재검증': compact, 'additional_api_calls':0,
               'full_document_complete':False, 'status':cache_replay_report.get('status')})
    print('상세 재검증 기록:', diagnostic_path)
else:
    print('기존 v1/D02/result.json이 없어 캐시 재검증을 건너뜁니다. 추가 API 호출: 0회')


{
  "기존_D02_캐시_재검증": [
    {
      "batch_id": "map-0002",
      "status": "cached_batch_revalidated",
      "valid": true,
      "chunk_count": 27,
      "claims_revalidated": 15,
      "additional_api_calls": 0,
      "restored_quotes": 7,
      "omitted_condition_gaps": 0
    }
  ],
  "additional_api_calls": 0,
  "full_document_complete": false,
  "status": null
}
상세 재검증 기록: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase5_local_20261111/phase5_outputs/final_validation/phase5_final_v3/cache_revalidation/D02.json


## 4. 개발 5건 — 검증 가능한 기존 결과 재사용 후 남은 문서 생성

기존 `phase5_outputs/runs/`의 결과는 현재 원문·청크·모델 요청 fingerprint가 모두 일치하고
전체 입력·인용 검증에 통과해야 재사용합니다. 기존 파일은 변경하지 않습니다.
인용 검사 실패나 API 오류가 발생하면 해당 결과와 진단을 저장하고 멈춥니다. 실패 결과도 자동 재시도하지 않습니다.
반복 Run All은 이미 기록된 사례를 다시 생성하지 않습니다. 수정 후 재검증은 새 `SUITE_ID`로 구분합니다.


In [6]:
def ensure_api_ready():
    try:
        sdk_version = version('openai')
    except PackageNotFoundError:
        raise RuntimeError('현재 커널에 OpenAI SDK가 없습니다. 이 커널의 Python으로 requirements-phase5.txt를 설치하고 커널을 Restart하세요.') from None
    if not os.environ.get('OPENAI_API_KEY', '').strip():
        from getpass import getpass
        key = getpass('OpenAI API 키 (표시되지 않음): ').strip()
        if not key:
            raise ValueError('API 키가 입력되지 않았습니다.')
        os.environ['OPENAI_API_KEY'] = key
        del key
    print('OpenAI SDK:', sdk_version, '/ 키 설정: True')

if RUN_DEV:
    reused = suite.import_existing('dev')
    print('이번에 재사용한 기존 결과:', reused)
    missing_dev = [s for s in suite.samples if s['split']=='dev'
        and not (suite.folder/s['sample_id']/'result.json').exists()]
    if ENABLE_OPENAI and missing_dev:
        ensure_api_ready()
    dev_execution = suite.run_stage('dev', enabled=ENABLE_OPENAI, reviewer=REVIEWER)
    show_json(dev_execution)
else:
    print('개발 생성 실행 옵션이 꺼져 있습니다. 저장된 결과의 검토·판정은 가능합니다.')


이번에 재사용한 기존 결과: []
OpenAI SDK: 3.49.0 / 키 설정: True
요약 실행: D02 서울특별시_2026_2027년 글로벌 패션브랜드 육성 용역_제안요청서.hwpx
실행: incomplete / 출처·전체 입력: False
{
  "status": "stopped_after_technical_failure",
  "sample_id": "D02",
  "errors": [
    "map-0003:source_citation_validation_failed"
  ],
  "error_detail": null,
  "error_diagnostics": null,
  "failed_checks": [
    "all_chunks_validated_once",
    "execution_completed",
    "no_batch_errors",
    "live_model_origin",
    "exact_model_requests_match"
  ],
  "automatic_retry": false
}


## 5. 실제 결과 검토 — 이 단계에서 원문과 비교

`phase5_outputs/final_validation/<SUITE_ID>/D01`부터 `D05`까지의 폴더에서
`answer.md`, `comparison.csv`, `technical_check.json`, `source_audit.json`을 확인하세요.

CSV의 **정확성·누락없음·조건보존** 세 열에만 `통과` 또는 `실패`를 적고, 이유는 `메모`에 씁니다.
처리한 입력에서 미확인·공고문 참조·상충 후보 표시도 원문과 맞으면 통과할 수 있습니다.
`추가원문확인후보`는 이미 처리한 원문에서 찾은 검토 후보이며 생성 답변에 자동으로 추가한 내용이 아닙니다.
예산의 부가세·상한·정산 조건, 기간의 안정화·기간 이후 의무, 제출의 서류/기한/방식을 함께 대조하세요.

문서별 11행 × 3판정 = 33판정입니다. 이미 대조한 동일한 결과는 기존 검토를 활용해 기록하면 됩니다.
기준·실제 요약·인용 열을 수정하면 검토 기록이 무효가 됩니다. Excel에서는 CSV 형식을 유지해 저장하세요.
요약에 잘못되거나 빠진 부분이 있으면 `실패`로 남겨주세요. 판정만 바꿔 통과시키지 않습니다.

검토 후 설정의 `REVIEWER`에 이름을 적고 아래 셀을 실행하세요. 생성 결과·검토 CSV는 덮어쓰지 않습니다.


In [7]:
report = suite.report(REVIEWER)
show_json({'개발5건_통과': report['dev_passed'], '사례별_판정': report['cases'][:5]})
print('기술 검사에 통과한 문서의 review_path에 CSV가 생성됩니다.')


{
  "개발5건_통과": false,
  "사례별_판정": [
    {
      "accepted": false,
      "technical_passed": true,
      "human_status": "review_pending",
      "technical_failures": [],
      "status": "review_pending",
      "sample_id": "D01",
      "split": "dev",
      "model_status": "partial",
      "execution_status": "completed",
      "validated_chunk_coverage": 1.0,
      "reviewed_judgements": 0,
      "judgements": 33,
      "source_audit_candidates": 3,
      "citation_gap_review_count": 0,
      "result_path": "/Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase5_local_20261111/phase5_outputs/final_validation/phase5_final_v3/D01/result.json",
      "review_path": "/Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase5_local_20261111/phase5_outputs/final_validation/phase5_final_v3/D01/comparison.csv"
    },
    {
      "accepted": false,
      "technical_passed": false,
      "human_status": "technical_failed_before_review",
      "technical_failures": [
        "all_

## 6. 최종 2건 — 개발 검토 통과 후 같은 코드·프롬프트로 실행

개발 5건의 내용 검토가 모두 통과해야 이 단계가 실행됩니다. 설정에서 `RUN_FINAL=True`로 바꾸고 Run All하세요.
이때 개발 5건은 저장된 결과를 재사용합니다. 코드·설정·기준표를 바꾸면 같은 검증 ID로 계속 진행할 수 없습니다.

최종 2건에서 문제를 발견하고 이를 이용해 수정하면 두 문서는 개선에 사용한 사례가 됩니다.
수정 전 최종 평가 기록을 남기고, 다음 최종 검증용 문서는 새로 정해야 합니다.


In [8]:
if RUN_FINAL:
    before_final = suite.report(REVIEWER)
    missing_final = [s for s in suite.samples if s['split']=='final'
        and not (suite.folder/s['sample_id']/'result.json').exists()]
    if ENABLE_OPENAI and before_final['dev_passed'] and missing_final:
        ensure_api_ready()
    final_execution = suite.run_stage('final', enabled=ENABLE_OPENAI, reviewer=REVIEWER)
    show_json(final_execution)
else:
    print('최종 2건 생성: 미실행. 개발 5건 검토를 통과한 뒤 RUN_FINAL=True로 바꾸세요.')


최종 2건 생성: 미실행. 개발 5건 검토를 통과한 뒤 RUN_FINAL=True로 바꾸세요.


## 7. 최종 판정과 공유할 결과 — API 호출 없음

R06·R07도 문서별 `comparison.csv`를 검토한 다음 이 셀만 실행하세요.
`phase5_sample_validation_passed=true`는 고정된 표본 7건·77항목의 요약 검증 통과입니다.
모든 99개 문서나 후속 질문·새 도구·전체 에이전트 성능까지 검증했다는 뜻은 아닙니다.

아래에서 만든 ZIP을 보내주면 문서별 결과·오류·검토 판정을 함께 확인할 수 있습니다.
ZIP은 이번 검증의 결과·인용·검토표·실행 기록만 포함합니다. API 키·환경변수·응답 캐시·전체 문서 집합은 포함하지 않습니다.


In [9]:
suite.assert_frozen()
final_report = suite.report(REVIEWER)
display(Markdown('| 표본 | 모델 상태 | 실행 상태 | 기술 검사 | 내용 검토 | 사례 통과 |\n'
    '| --- | --- | --- | --- | --- | --- |\n' + '\n'.join(
    f"| {c['sample_id']} | {c.get('model_status','미실행')} | {c.get('execution_status','미실행')} | {c.get('technical_passed')} | {c.get('human_status','미검토')} | {c['accepted']} |"
    for c in final_report['cases'])))
show_json({k: final_report[k] for k in ['status','phase5_sample_validation_passed',
    'dev_passed','final_passed','offline_failure_checks_passed','budget_this_suite','scope']})
result_zip = suite.export(REVIEWER)
print('공유할 결과 ZIP:', result_zip)
print('상세 판정:', suite.folder / 'final_report.json')
if final_report['phase5_sample_validation_passed']:
    print('Phase 5 표본 7건 검증: 통과. 이 범위의 검증 기록을 바탕으로 다음 단계로 진행할 수 있습니다.')
else:
    print('아직 미실행·미검토 또는 실패한 항목이 있습니다. 사례별 판정을 확인하세요.')


| 표본 | 모델 상태 | 실행 상태 | 기술 검사 | 내용 검토 | 사례 통과 |
| --- | --- | --- | --- | --- | --- |
| D01 | partial | completed | True | review_pending | False |
| D02 | partial | incomplete | False | technical_failed_before_review | False |
| D03 | 미실행 | 미실행 | False | 미검토 | False |
| D04 | 미실행 | 미실행 | False | 미검토 | False |
| D05 | 미실행 | 미실행 | False | 미검토 | False |
| R06 | 미실행 | 미실행 | False | 미검토 | False |
| R07 | 미실행 | 미실행 | False | 미검토 | False |

{
  "status": "validation_pending_or_failed",
  "phase5_sample_validation_passed": false,
  "dev_passed": false,
  "final_passed": false,
  "offline_failure_checks_passed": true,
  "budget_this_suite": {
    "api_calls": 1,
    "attempted_input_bytes": 34043
  },
  "scope": "고정된 공개 RFP 7건·77항목의 요약 검증. 전체 99건·Q&A 전체 성능을 확정하지 않습니다."
}
공유할 결과 ZIP: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase5_local_20261111/phase5_outputs/final_validation/phase5_final_v3_results.zip
상세 판정: /Users/codeit/Desktop/workspace_mac/중급 프로젝트/rfp_phase5_local_20261111/phase5_outputs/final_validation/phase5_final_v3/final_report.json
아직 미실행·미검토 또는 실패한 항목이 있습니다. 사례별 판정을 확인하세요.
